## InternVideo2 — Video + Text Shared Semantic Space

Focused experiment: shared semantic space between **video** and **text** using
InternVideo2 Stage2 as the sole encoder.

### Architecture
```
MP4 ──► segment into clips ──► InternVideo2 video encoder ──► Qdrant  iv2_only_<video-hash>
                │
                └──► ffmpeg audio ──► Whisper ──► transcript (LLM context only)

text query ──► BERT-large (IV2 text encoder) ──► similarity search ──► clips
                                                          │
                                                 attach transcript ──► LLM
```

### API notes
The HuggingFace checkpoint `OpenGVLab/InternVideo2-Stage2_1B-224p-f4` ships a
custom `modeling_internvideo2.py` loaded via `trust_remote_code=True`.
The actual methods are:
- **`model.get_vid_feat(tensor)`** — video encoder, input shape `(B, C, T, H, W)`
- **`model.get_txt_feat(input_ids, attention_mask, token_type_ids)`** — BERT-large text encoder
- Helper imports: `vid2tensor`, `_frame_from_video`, `retrieve_text` from `modeling_internvideo2`

> **Verified checkpoints on HuggingFace (Stage2, retrieval-optimised):**
> - `OpenGVLab/InternVideo2-Stage2_1B-224p-f4` — 1B, 4 frames, **default**
> - `OpenGVLab/InternVideo2-Stage2_1B-224p-f8` — 1B, 8 frames, better temporal resolution
> - `OpenGVLab/InternVideo2-Stage2_6B-224p-f4` — 6B, ~12 GB VRAM in fp16
> - `OpenGVLab/InternVideo2-CLIP-1B-224p-f8` — CLIP-style contrastive head variant


### 1. Configuration


In [1]:
import hashlib, os, warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv("../setup.env", override=True)

# Reuse the Hugging Face cache downloaded on Windows when this notebook runs
# inside WSL. The WSL virtualenv also installs this behavior at Python startup
# via install_wsl_hf_cache_hook.py, but this fallback helps with other kernels.
def _truthy_env(name: str, default: bool = False) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() not in {"0", "false", "no", "off"}


def _running_in_wsl() -> bool:
    if os.getenv("WSL_DISTRO_NAME"):
        return True
    try:
        import platform
        return "microsoft" in platform.release().lower()
    except Exception:
        return False


def _configure_wsl_hf_cache() -> None:
    if not _running_in_wsl():
        return
    hf_home = os.getenv("WSL_WINDOWS_HF_HOME", "/mnt/c/Users/danie/.cache/huggingface")
    if not os.path.isdir(hf_home):
        print(f"  [warn] WSL Windows Hugging Face cache not found: {hf_home}")
        return
    force_cache = _truthy_env("MULTIRAG_FORCE_WSL_HF_CACHE", True)

    def set_cache_var(name: str, value: str) -> None:
        if force_cache or not os.getenv(name):
            os.environ[name] = value

    set_cache_var("HF_HOME", hf_home)
    set_cache_var("HF_HUB_CACHE", os.path.join(hf_home, "hub"))
    set_cache_var("HF_DATASETS_CACHE", os.path.join(hf_home, "datasets"))

    if _truthy_env("MULTIRAG_HF_OFFLINE", True) and not _truthy_env("MULTIRAG_HF_ALLOW_DOWNLOADS", False):
        os.environ.setdefault("HF_HUB_OFFLINE", "1")
        os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

    print(
        f"  Hugging Face cache: {os.environ['HF_HOME']} "
        f"(Windows cache mounted in WSL, offline={os.environ.get('HF_HUB_OFFLINE') == '1'})"
    )


_configure_wsl_hf_cache()

# ── Input ─────────────────────────────────────────────────────────────────────
VIDEO_PATH = os.getenv("VIDEO_PATH", "./content/video.mp4")


def _video_source_id(path: str, length: int = 12) -> str:
    """Return a stable fingerprint so indexes from different videos never mix."""
    if not os.path.exists(path):
        return "missing"
    digest = hashlib.md5()
    with open(path, "rb") as video_file:
        for chunk in iter(lambda: video_file.read(65536), b""):
            digest.update(chunk)
    return digest.hexdigest()[:length]


VIDEO_SOURCE_ID = _video_source_id(VIDEO_PATH)

# ── InternVideo2 Stage2 ───────────────────────────────────────────────────────
# Only Stage2 checkpoints expose shared video-text embedding space.
# "OpenGVLab/InternVideo2-Stage2_1B-224p-f4"   — 1B, 4 frames (default)
# "OpenGVLab/InternVideo2-Stage2_1B-224p-f8"   — 1B, 8 frames
# "OpenGVLab/InternVideo2-Stage2_6B-224p-f4"   — 6B (~12 GB VRAM fp16)
# "OpenGVLab/InternVideo2-CLIP-1B-224p-f8"     — CLIP-style head
IV2_MODEL       = os.getenv("IV2_MODEL", "OpenGVLab/InternVideo2-Stage2_1B-224p-f4")
IV2_NUM_FRAMES  = int(os.getenv("IV2_NUM_FRAMES",  "8"))   # must match checkpoint suffix -f4 / -f8
IV2_SEGMENT_SECS= int(os.getenv("IV2_SEGMENT_SECS","8"))  # seconds per video clip
IV2_OVERLAP_SECS= int(os.getenv("IV2_OVERLAP_SECS", "2"))  # overlap between clips
IV2_BATCH_SIZE  = int(os.getenv("IV2_BATCH_SIZE",   "4"))  # clips per forward pass

# ── BERT tokenizer (text encoder of InternVideo2 Stage2) ─────────────────────
# Stage2 uses bert-large-uncased as text encoder — NOT the IV2 tokenizer.
BERT_MODEL      = "bert-large-uncased"
TEXT_MAX_LENGTH = int(os.getenv("TEXT_MAX_LENGTH", "77"))

# ── Whisper (transcript for LLM context only — not used for retrieval) ────────
WHISPER_MODEL    = os.getenv("WHISPER_MODEL",    "openai/whisper-large-v3")
WHISPER_LANGUAGE = os.getenv("WHISPER_LANGUAGE", "en")
SAMPLE_RATE      = 16000

# ── Retrieval ─────────────────────────────────────────────────────────────────
RETRIEVER_K    = int(os.getenv("RETRIEVER_K",    "8"))
RERANKER_TOP_N = int(os.getenv("RERANKER_TOP_N", "4"))
RERANKER_MODEL = os.getenv("RERANKER_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")
ENABLE_RERANKING = os.getenv("ENABLE_RERANKING", "true").lower() == "true"

# ── Generation ────────────────────────────────────────────────────────────────
GENERATION_MODEL   = os.getenv("GENERATION_MODEL",   "Qwen/Qwen2.5-3B-Instruct")
GENERATION_BACKEND = os.getenv("GENERATION_BACKEND", "hf")  # "hf" | "ollama"
GENERATION_MAX_NEW_TOKENS = int(os.getenv("GENERATION_MAX_NEW_TOKENS", "512"))
HF_DEVICE_MAP  = os.getenv("HF_DEVICE_MAP",  "auto")
HF_TORCH_DTYPE = os.getenv("HF_TORCH_DTYPE", "auto")

# ── Qdrant ────────────────────────────────────────────────────────────────────
QDRANT_URL       = os.getenv("QDRANT_URL",     "http://localhost:6333")
QDRANT_API_KEY   = os.getenv("QDRANT_API_KEY", "")
QDRANT_IV2_ONLY  = os.getenv("QDRANT_IV2_ONLY", f"iv2_only_{VIDEO_SOURCE_ID}")
RESET_COLLECTION = os.getenv("RESET_COLLECTION", "false").lower() == "true"

# ── Persistence + cross-notebook results ──────────────────────────────────────
PERSIST_DIR   = os.getenv("PERSIST_DIR",   "./cache/iv2_only/")
RESULTS_AUDIO = os.getenv("RESULTS_AUDIO", "./cache/audio/audio_eval_results.json")
RESULTS_FUSION= os.getenv("RESULTS_FUSION","./cache/multimodal/multimodal_eval_results.json")
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs("./content/", exist_ok=True)

print("Configuration loaded.")
print(f"  Video      : {VIDEO_PATH} (source={VIDEO_SOURCE_ID})")
print(f"  IV2 model  : {IV2_MODEL}")
print(f"  Frames/clip: {IV2_NUM_FRAMES}  | seg={IV2_SEGMENT_SECS}s | overlap={IV2_OVERLAP_SECS}s")
print(f"  BERT       : {BERT_MODEL}")
print(f"  Whisper    : {WHISPER_MODEL}")
print(f"  Generator  : {GENERATION_MODEL} ({GENERATION_BACKEND})")
print(f"  Qdrant     : {QDRANT_URL}  collection={QDRANT_IV2_ONLY}")


Configuration loaded.
  Video      : ./content/video.mp4 (source=467e468b2f83)
  IV2 model  : OpenGVLab/InternVideo2-Stage2_1B-224p-f4
  Frames/clip: 8  | seg=8s | overlap=2s
  BERT       : bert-large-uncased
  Whisper    : openai/whisper-large-v3
  Generator  : mistral-nemo:latest (ollama)
  Qdrant     : http://localhost:6333  collection=iv2_only_467e468b2f83


### 2. Audio Extraction + Whisper Transcription

Whisper is used **only** for LLM context — not for retrieval.
Results are cached to avoid re-running on long videos.

FFMPEG is mandatory:

sudo apt install -y ffmpeg

In [2]:
import torch, numpy as np, json, hashlib, librosa
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline

def _file_hash(path: str, n: int = 12) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""): h.update(chunk)
    return h.hexdigest()[:n]

# ── Extract audio track ───────────────────────────────────────────────────────
audio_track = Path(PERSIST_DIR) / f"audio_{_file_hash(VIDEO_PATH)}.wav"
if not audio_track.exists():
    print("Extracting audio track ...")
    rc = os.system(f'ffmpeg -y -i "{VIDEO_PATH}" -ar {SAMPLE_RATE} -ac 1 '
                   f'"{audio_track}" -loglevel error')
    if rc != 0 or not audio_track.exists():
        raise RuntimeError("ffmpeg failed. Verify ffmpeg is installed and VIDEO_PATH is valid.")
    print(f"  ✓ Saved: {audio_track.name}")
else:
    print(f"✓ Audio cached: {audio_track.name}")

# ── Whisper transcription ─────────────────────────────────────────────────────
transcript_cache = Path(PERSIST_DIR) / f"transcript_{_file_hash(VIDEO_PATH)}_{WHISPER_MODEL.split('/')[-1]}.json"

if transcript_cache.exists():
    with open(transcript_cache) as f:
        transcript_data = json.load(f)
else:
    print(f"Loading Whisper ({WHISPER_MODEL}) ...")
    _w_device = "cuda" if torch.cuda.is_available() else "cpu"
    _w_dtype  = torch.float16 if _w_device == "cuda" else torch.float32
    _w_proc   = AutoProcessor.from_pretrained(WHISPER_MODEL)
    _w_model  = AutoModelForSpeechSeq2Seq.from_pretrained(
        WHISPER_MODEL, torch_dtype=_w_dtype, low_cpu_mem_usage=True).to(_w_device)
    _w_pipe   = hf_pipeline(
        "automatic-speech-recognition",
        model=_w_model, tokenizer=_w_proc.tokenizer,
        feature_extractor=_w_proc.feature_extractor,
        torch_dtype=_w_dtype, device=_w_device,
        return_timestamps=True, chunk_length_s=30, batch_size=16,
        generate_kwargs={"language": WHISPER_LANGUAGE} if WHISPER_LANGUAGE else {},
    )
    waveform, _ = librosa.load(str(audio_track), sr=SAMPLE_RATE, mono=True)
    print(f"  Transcribing {len(waveform)/SAMPLE_RATE:.0f}s ...")
    result = _w_pipe(waveform.copy(), return_timestamps=True)
    transcript_data = {
        "text":   result["text"],
        "chunks": [{"text": c["text"], "timestamp": list(c["timestamp"])}
                   for c in result.get("chunks", [])],
    }
    with open(transcript_cache, "w") as f:
        json.dump(transcript_data, f, ensure_ascii=False, indent=2)
    del _w_model, _w_pipe, _w_proc
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"  ✓ Transcript saved ({len(transcript_data['chunks'])} segments).")

video_transcript_segs = transcript_data["chunks"]
print(f"\nFirst 400 chars: {transcript_data['text'][:400]}")


✓ Audio cached: audio_467e468b2f83.wav

First 400 chars:  How did a single paper, attention is all you need, reshape the entire AI landscape? In this video, we will unpack the transformer architecture. We will see how it works, what makes it so powerful, and why it replaced almost every older neural network design. Before diving in, let's take a quick step back. The goal of machine learning is to learn a mapping from inputs to outputs. For example, in p


### 3. InternVideo2 Stage2 — Model Loading

**Important API notes** verified against the official HuggingFace model cards:

- `AutoModel.from_pretrained(..., trust_remote_code=True)` downloads the custom
  `modeling_internvideo2.py` from HF and registers it. This file exposes the helper
  functions `vid2tensor`, `_frame_from_video`, `retrieve_text`.
- **Video encoder**: `model.get_vid_feat(video_tensor)` — input shape `(B, C, T, H, W)`.
- **Text encoder**: `model.get_txt_feat(input_ids, attention_mask, token_type_ids)` —
  the text encoder is **BERT-large** (`bert-large-uncased`), loaded separately.
- The embedding dim for both encoders is **1024** (Stage2 1B checkpoint).


In [3]:
# !git clone https://github.com/OpenGVLab/InternVideo.git


In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, CLIPTokenizerFast

IV2_MODEL = "OpenGVLab/InternVideo2_CLIP_S"
IV2_TOKENIZER_MODEL = "openai/clip-vit-base-patch16"

IV2_AVAILABLE = False
IV2_DIM       = 512
iv2_model     = None
iv2_tokenizer = None

print(f"Loading InternVideo2: {IV2_MODEL} ...")

_iv2_device = "cuda" if torch.cuda.is_available() else "cpu"
_iv2_dtype  = torch.float32

try:
    iv2_model = AutoModel.from_pretrained(
        IV2_MODEL,
        torch_dtype=_iv2_dtype,
        trust_remote_code=True,
    ).to(_iv2_device).eval()

    # InternVideo2_CLIP_S has CLIP-like text config, but does not expose
    # a tokenizer compatible with AutoTokenizer.
    iv2_tokenizer = CLIPTokenizerFast.from_pretrained(IV2_TOKENIZER_MODEL)

    # ── Verify embedding dim ─────────────────────────────────────────────────
    _probe = iv2_tokenizer(
        ["probe"],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    ).to(_iv2_device)

    def _iv2_encode_text(tokenized):
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized.get("attention_mask", None)
        token_type_ids = tokenized.get("token_type_ids", None)

        if hasattr(iv2_model, "get_text_features"):
            return iv2_model.get_text_features(**tokenized)

        if hasattr(iv2_model, "get_txt_feat"):
            # Try the most common InternVideo2 signatures.
            try:
                if token_type_ids is not None:
                    return iv2_model.get_txt_feat(
                        input_ids,
                        attention_mask,
                        token_type_ids,
                    )
                return iv2_model.get_txt_feat(
                    input_ids,
                    attention_mask,
                )
            except TypeError:
                try:
                    return iv2_model.get_txt_feat(input_ids)
                except TypeError:
                    return iv2_model.get_txt_feat(
                        text=input_ids,
                        attention_mask=attention_mask,
                    )

        if hasattr(iv2_model, "encode_text"):
            try:
                return iv2_model.encode_text(input_ids)
            except TypeError:
                return iv2_model.encode_text(
                    input_ids,
                    attention_mask=attention_mask,
                )

        raise AttributeError(
            "No known text embedding method found. "
            "Try: [m for m in dir(iv2_model) "
            "if 'text' in m.lower() or 'txt' in m.lower() "
            "or 'encode' in m.lower() or 'feat' in m.lower()]"
        )

    with torch.no_grad():
        _txt_feat = _iv2_encode_text(_probe)
        _txt_feat = F.normalize(_txt_feat.float(), dim=-1)

    IV2_DIM = _txt_feat.shape[-1]

    n_params = sum(p.numel() for p in iv2_model.parameters()) / 1e6
    IV2_AVAILABLE = True

    print(
        f"  ✓ InternVideo2 ready | "
        f"device={_iv2_device} | "
        f"dim={IV2_DIM} | "
        f"params={n_params:.0f}M"
    )
    print(f"  ✓ Tokenizer ready: {IV2_TOKENIZER_MODEL}")

except AttributeError as e:
    print(f"  ✗ Method not found: {e}")
    if iv2_model is not None:
        print("Available likely methods:")
        print([
            m for m in dir(iv2_model)
            if "text" in m.lower()
            or "txt" in m.lower()
            or "encode" in m.lower()
            or "feat" in m.lower()
        ])
    IV2_AVAILABLE = False

except Exception as e:
    print(f"  ✗ Loading failed: {e}")
    print("  → Try: pip install decord timm einops flash-attn")
    IV2_AVAILABLE = False


Loading InternVideo2: OpenGVLab/InternVideo2_CLIP_S ...
  ✗ Loading failed: No module named 'flash_attn'
  → Try: pip install decord timm einops flash-attn


In [5]:
# ── Inspect available feature methods (run if loading raised AttributeError) ──
# This cell helps discover the correct method names for a given checkpoint variant.
if iv2_model is not None:
    feat_methods = [m for m in dir(iv2_model) if "feat" in m.lower() or "encode" in m.lower()]
    print("Available feature/encode methods on this checkpoint:")
    for m in feat_methods:
        print(f"  {m}")


### 4. Encoding Functions

`encode_text_iv2` uses the BERT-large tokenizer and `model.get_txt_feat`.  
`encode_video_segment` samples frames uniformly from `[start_sec, end_sec]` with decord,
formats them as `(1, C, T, H, W)` (the tensor shape expected by `model.get_vid_feat`),
and returns an L2-normalised embedding.


In [6]:
import decord
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any, Tuple
import uuid

# ── Frame transform matching InternVideo2's vid2tensor preprocessing ──────────
# Source: modeling_internvideo2.py bundled in the HF repo.
_IV2_MEAN = (0.485, 0.456, 0.406)
_IV2_STD  = (0.229, 0.224, 0.225)
_iv2_transform = transforms.Compose([
    transforms.Lambda(lambda x: x.convert("RGB") if hasattr(x, "convert") else x),
    transforms.Resize(224, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=_IV2_MEAN, std=_IV2_STD),
])


import numpy as np
import torch
import torch.nn.functional as F
from typing import List

TEXT_MAX_LENGTH = 77

def encode_text_iv2(texts: List[str]) -> np.ndarray:
    """
    Encode text with InternVideo2 text encoder.
    Returns:
        (N, IV2_DIM) float32 numpy array
    """

    if not IV2_AVAILABLE:
        raise RuntimeError("InternVideo2 not loaded.")

    toks = iv2_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=TEXT_MAX_LENGTH,
    ).to(_iv2_device)

    with torch.no_grad():

        if hasattr(iv2_model, "get_text_features"):
            feat = iv2_model.get_text_features(**toks)

        elif hasattr(iv2_model, "get_txt_feat"):
            try:
                feat = iv2_model.get_txt_feat(
                    toks["input_ids"],
                    toks.get("attention_mask", None),
                    toks.get("token_type_ids", None),
                )
            except TypeError:
                feat = iv2_model.get_txt_feat(
                    toks["input_ids"],
                    toks.get("attention_mask", None),
                )

        elif hasattr(iv2_model, "encode_text"):
            feat = iv2_model.encode_text(toks["input_ids"])

        else:
            raise AttributeError(
                f"No compatible text encoder found.\n"
                f"Available methods: {[m for m in dir(iv2_model) if 'text' in m.lower() or 'txt' in m.lower()]}"
            )

        feat = F.normalize(feat.float(), dim=-1)

    return feat.cpu().numpy()


def _sample_frames_pil(video_path: str, start_sec: float, end_sec: float,
                        num_frames: int) -> Optional[List]:
    """Sample `num_frames` PIL frames uniformly from [start_sec, end_sec]."""
    from PIL import Image as PILImage
    try:
        vr = decord.VideoReader(video_path, ctx=decord.cpu(0))
        fps   = vr.get_avg_fps()
        total = len(vr)
        s = min(int(start_sec * fps), total - 1)
        e = min(int(end_sec   * fps), total - 1)
        if e <= s: e = min(s + 1, total - 1)
        idx    = np.clip(np.linspace(s, e, num_frames, dtype=int), 0, total - 1)
        frames = vr.get_batch(idx).asnumpy()   # (T, H, W, 3) uint8
        return [PILImage.fromarray(f) for f in frames]
    except Exception as ex:
        print(f"  [sample_frames] {ex}")
        return None


def encode_video_segment(
    video_path: str,
    start_sec: float,
    end_sec: float,
    num_frames: int = IV2_NUM_FRAMES,
) -> np.ndarray:

    pil_frames = _sample_frames_pil(video_path, start_sec, end_sec, num_frames)

    if pil_frames is None:
        return np.zeros(IV2_DIM, dtype=np.float32)

    frame_tensors = torch.stack(
        [_iv2_transform(f) for f in pil_frames]
    )  # (T, 3, 224, 224)

    # IMPORTANT: InternVideo2 expects (B,T,C,H,W)
    video_tensor = frame_tensors.unsqueeze(0)

    model_dtype = next(iv2_model.parameters()).dtype
    video_tensor = video_tensor.to(
        device=_iv2_device,
        dtype=model_dtype
    )

    with torch.no_grad():
        feat = iv2_model.encode_vision(video_tensor)
        feat = torch.nn.functional.normalize(
            feat.float(),
            dim=-1
        )

    return feat.cpu().numpy()[0]


In [7]:
# ── Quick sanity check ────────────────────────────────────────────────────────
# Verify that text and video embeddings are in the same space
# (cosine similarity between a relevant text and a video clip should be > 0)
if IV2_AVAILABLE:
    test_texts = [
        "attention mechanism transformer architecture",
        "a cat sleeping on a sofa",   # irrelevant control
    ]
    txt_vecs = encode_text_iv2(test_texts)
    vid_vec  = encode_video_segment(VIDEO_PATH, start_sec=0, end_sec=IV2_SEGMENT_SECS)

    sims = txt_vecs @ vid_vec   # dot product = cosine sim (L2-normalised)
    print("Text-video cosine similarity (sanity check):")
    for t, s in zip(test_texts, sims):
        print(f"  {s:+.4f}  {t!r}")
    print("\nExpected: first text more similar than control phrase.")
else:
    print("IV2 not available — skipping sanity check.")


IV2 not available — skipping sanity check.


### 5. Video Segmentation + Qdrant Indexing


In [8]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

@dataclass
class VideoSegment:
    """Fixed-length video segment with Whisper transcript for LLM context."""
    start_sec:   float
    end_sec:     float
    transcript:  str  = ""
    source_file: str  = ""

    @property
    def timestamp_label(self) -> str:
        def fmt(s): m, sec = divmod(int(s), 60); return f"{m:02d}:{sec:02d}"
        return f"[{fmt(self.start_sec)} – {fmt(self.end_sec)}]"

    @property
    def content(self) -> str:
        return self.transcript   # alias for metric functions

    @property
    def doc_type(self) -> str:
        return "text"

def get_transcript_for_range(segs: List[dict], start_s: float, end_s: float,
                              padding: float = 1.0) -> str:
    total = max((s["timestamp"][1] or 0) for s in segs) if segs else 0
    return " ".join(
        s["text"] for s in segs
        if (s["timestamp"][1] or total) >= (start_s - padding)
        and (s["timestamp"][0] or 0)     <= (end_s   + padding)
    ).strip()

# ── Qdrant setup ──────────────────────────────────────────────────────────────
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)

if RESET_COLLECTION and client.collection_exists(QDRANT_IV2_ONLY):
    client.delete_collection(QDRANT_IV2_ONLY)
    print(f"✓ Deleted '{QDRANT_IV2_ONLY}'")

if not client.collection_exists(QDRANT_IV2_ONLY):
    client.create_collection(
        collection_name = QDRANT_IV2_ONLY,
        vectors_config  = VectorParams(size=IV2_DIM, distance=Distance.COSINE),
    )
    print(f"✓ Created '{QDRANT_IV2_ONLY}' (dim={IV2_DIM})")
else:
    print(f"✓ Existing '{QDRANT_IV2_ONLY}' (dim={IV2_DIM})")

iv2_docstore: Dict[str, VideoSegment] = {}


✓ Existing 'iv2_only_467e468b2f83' (dim=512)


In [ ]:
def index_video(video_path: str, batch_size: int = IV2_BATCH_SIZE) -> List[str]:
    """
    Segment the video, encode each clip with InternVideo2 video encoder,
    attach Whisper transcript, and upsert to Qdrant.
    """
    if not IV2_AVAILABLE:
        print("InternVideo2 not available.")
        return []

    vr = decord.VideoReader(video_path, ctx=decord.cpu(0))
    duration = len(vr) / vr.get_avg_fps()

    print(f"Video: {Path(video_path).name} | {duration:.1f}s")

    step = IV2_SEGMENT_SECS - IV2_OVERLAP_SECS
    ranges, t = [], 0.0

    while t < duration:
        e = min(t + IV2_SEGMENT_SECS, duration)
        ranges.append((t, e))

        if e >= duration:
            break

        t += step

    print(
        f"Segments: {len(ranges)} × ~{IV2_SEGMENT_SECS}s "
        f"(overlap={IV2_OVERLAP_SECS}s)"
    )

    all_ids, points = [], []

    for i in range(0, len(ranges), batch_size):
        batch = ranges[i : i + batch_size]

        for start_s, end_s in batch:
            try:
                vec = encode_video_segment(video_path, start_s, end_s)
                vec = np.asarray(vec, dtype=np.float32)

                if vec.shape[-1] != IV2_DIM:
                    raise ValueError(
                        f"Bad vector dim: got {vec.shape[-1]}, expected {IV2_DIM}"
                    )

            except Exception as e:
                print(f"  ✗ [{start_s:.0f}s → {end_s:.0f}s]: {e}")
                vec = np.zeros(IV2_DIM, dtype=np.float32)

            transcript = get_transcript_for_range(
                video_transcript_segs,
                start_s,
                end_s,
            )

            seg = VideoSegment(
                start_sec=start_s,
                end_sec=end_s,
                transcript=transcript,
                source_file=video_path,
            )

            uid = str(uuid.uuid5(
                uuid.NAMESPACE_URL,
                f"{VIDEO_SOURCE_ID}:iv2:{start_s:.3f}:{end_s:.3f}",
            ))
            iv2_docstore[uid] = seg
            all_ids.append(uid)

            points.append(
                PointStruct(
                    id=uid,
                    vector=vec.tolist(),
                    payload={
                        "start_sec": float(start_s),
                        "end_sec": float(end_s),
                        "timestamp": seg.timestamp_label,
                        "preview": transcript[:120],
                        "transcript": transcript,
                        "source_file": str(video_path),
                    },
                )
            )

        prog = min(i + batch_size, len(ranges))
        print(f"  [{prog}/{len(ranges)}] encoded")

    for i in range(0, len(points), 100):
        client.upsert(
            collection_name=QDRANT_IV2_ONLY,
            points=points[i : i + 100],
        )

    print(f"\n✓ Indexed {len(all_ids)} segments.")
    return all_ids


iv2_ids = index_video(VIDEO_PATH)


InternVideo2 not available.


: 

### 6. Retrieval (text → video)


In [ ]:
from sentence_transformers import CrossEncoder

if ENABLE_RERANKING:
    print(f"Loading cross-encoder: {RERANKER_MODEL} ...")
    try:
        reranker = CrossEncoder(RERANKER_MODEL)
        RERANKER_AVAILABLE = True
        print("  ✓ Reranker ready.")
    except Exception as e:
        print(f"  ✗ {e}")
        reranker = None; RERANKER_AVAILABLE = False
else:
    reranker = None; RERANKER_AVAILABLE = False
    print("Reranking disabled.")


def iv2_retrieve(query: str) -> List[VideoSegment]:
    """
    Retrieve relevant video segments for a text query.
    1. Encode query with BERT-large (IV2 text encoder) → query vector in IV2 space.
    2. Cosine similarity search against video segment embeddings in Qdrant.
    3. Cross-encoder reranking on Whisper transcript of retrieved segments.
    """
    if not IV2_AVAILABLE:
        return []

    try:
        qvec = encode_text_iv2([query])[0].tolist()
        hits = client.query_points(
            collection_name=QDRANT_IV2_ONLY,
            query=qvec, limit=RETRIEVER_K, with_payload=True,
        ).points
    except Exception as e:
        print(f"[iv2_retrieve] {e}"); return []

    candidates: List[Tuple[str, VideoSegment]] = []
    for h in hits:
        seg = iv2_docstore.get(str(h.id))
        if seg:
            candidates.append((seg.transcript[:500], seg))

    if not candidates:
        return []

    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        scores = reranker.predict([(query, t) for t, _ in candidates])
        ranked = sorted(zip(scores, [s for _, s in candidates]),
                        key=lambda x: x[0], reverse=True)
        return [s for _, s in ranked[:RERANKER_TOP_N]]
    return [s for _, s in candidates[:RERANKER_TOP_N]]


# Sanity check
test = iv2_retrieve("What is the attention mechanism in Transformers?")
print(f"\nRetrieved {len(test)} segments:")
for i, s in enumerate(test):
    print(f"  [{i}] {s.timestamp_label} | {s.transcript[:100]!r}")


### 7. Generation (text-only LLM)


In [ ]:
import subprocess

def get_windows_host_ip():
    try:
        ip = subprocess.check_output(
            "ip route | awk '/default/ {print $3}'",
            shell=True,
            text=True
        ).strip()
        return ip
    except Exception as e:
        raise RuntimeError(f"Could not determine Windows host IP: {e}")

WIN_HOST_IP = get_windows_host_ip()
OLLAMA_BASE_URL = f"http://{WIN_HOST_IP}:11434"

print("Windows host IP:", WIN_HOST_IP)
print("Ollama URL:", OLLAMA_BASE_URL)


Windows host IP: 172.28.16.1
Ollama URL: http://172.28.16.1:11434


In [ ]:
import gc, os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GENERATOR_AVAILABLE = False
_gen_tok = _gen_model = _ollama_llm = None
USE_OLLAMA = False


def _int_env(name: str, default: int) -> int:
    try:
        return int(os.getenv(name, str(default)))
    except (TypeError, ValueError):
        return default


OLLAMA_NUM_CTX = _int_env("OLLAMA_NUM_CTX", 4096)
OLLAMA_NUM_PREDICT = _int_env("OLLAMA_NUM_PREDICT", 512)
OLLAMA_KEEP_ALIVE = os.getenv("OLLAMA_KEEP_ALIVE", "0s")
OLLAMA_NUM_GPU = os.getenv("OLLAMA_NUM_GPU", "").strip()


def _ollama_runtime_kwargs() -> dict:
    kwargs = {
        "num_ctx": OLLAMA_NUM_CTX,
        "num_predict": OLLAMA_NUM_PREDICT,
        "keep_alive": OLLAMA_KEEP_ALIVE,
    }
    if OLLAMA_NUM_GPU:
        kwargs["num_gpu"] = int(OLLAMA_NUM_GPU)
    return kwargs


def _release_torch_cache() -> None:
    gc.collect()
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def _invoke_ollama_generator(prompt: str) -> str:
    _release_torch_cache()
    try:
        llm = globals().get("ollama_llm") or globals().get("_ollama_llm")
        if llm is None:
            return "[Ollama generator unavailable]"
        return llm.invoke(prompt).content.strip()
    except Exception as exc:
        message = str(exc)
        if "requires more system memory" in message.lower():
            return (
                "[Ollama unavailable: not enough system RAM to load the generator. "
                "Restart idle notebook kernels, close other models, lower OLLAMA_NUM_CTX, "
                "or choose a smaller GENERATION_MODEL.]"
            )
        raise
if GENERATION_BACKEND == "ollama":
    try:
        from langchain_ollama import ChatOllama
        _ollama_llm = ChatOllama(
            model=GENERATION_MODEL,
            base_url=OLLAMA_BASE_URL,
            temperature=0,
            **_ollama_runtime_kwargs(),
        )
        USE_OLLAMA = True; GENERATOR_AVAILABLE = True
        print(f"✓ Ollama: {GENERATION_MODEL}")
    except Exception as e:
        print(f"✗ Ollama: {e}")
else:
    print(f"Loading generator: {GENERATION_MODEL} ...")
    try:
        _gen_tok   = AutoTokenizer.from_pretrained(GENERATION_MODEL, trust_remote_code=True)
        _gen_model = AutoModelForCausalLM.from_pretrained(
            GENERATION_MODEL, torch_dtype=HF_TORCH_DTYPE,
            device_map=HF_DEVICE_MAP, trust_remote_code=True).eval()
        GENERATOR_AVAILABLE = True
        print(f"  ✓ Generator ready.")
    except Exception as e:
        print(f"  ✗ {e}")


def generate_answer(segs: List[VideoSegment], question: str) -> str:
    if not GENERATOR_AVAILABLE:
        return "[Generator unavailable]"
    parts = [f"{s.timestamp_label}\n{s.transcript.strip()}"
             for s in segs if s.transcript.strip()]
    context = "\n\n".join(parts) if parts else "[No transcript context retrieved]"
    prompt = (
        "Answer the question using only the provided transcript excerpts. "
        "Each excerpt is preceded by its video timestamp. "
        "If the context is insufficient, state that explicitly.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    if USE_OLLAMA:
        return _invoke_ollama_generator(prompt)
    messages = [{"role": "user", "content": prompt}]
    if hasattr(_gen_tok, "apply_chat_template"):
        text_in = _gen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text_in = prompt
    inputs = _gen_tok([text_in], return_tensors="pt").to(_gen_model.device)
    with torch.no_grad():
        out = _gen_model.generate(**inputs, max_new_tokens=GENERATION_MAX_NEW_TOKENS,
                                  do_sample=False, temperature=None,
                                  pad_token_id=_gen_tok.eos_token_id)
    return _gen_tok.batch_decode(out[:, inputs.input_ids.shape[1]:],
                                 skip_special_tokens=True)[0].strip()


# Demo
q = "How do queries, keys, and values update token representations in attention?"
docs   = iv2_retrieve(q)
answer = generate_answer(docs, q)
print("=" * 70)
print(f"Q: {q}\n\nA: {answer}\n\nSources:")
for s in docs:
    print(f"  {s.timestamp_label} {s.transcript[:80]!r}")


✓ Ollama: mistral-nemo:latest
Q: How do queries, keys, and values update token representations in attention?

A: Based on the provided transcript excerpts, here's how queries, keys, and values update token representations in attention:

- **Queries**: They ask "what am I looking for?" or implicitly ask "what concept am I referring to?"
- **Keys**: They contain information about "what I have" or represent what is already known.
- **Values**: They carry the actual content to share. When a query matches with relevant keys, the corresponding values are used to update the token representations.

So, in essence, queries help identify which tokens (through their associated keys) should be considered for updating the representation of a given token, and the values provide the new information to incorporate into that updated representation.

Sources:
  [06:00 – 06:08] 'At each step, attention mixes information across tokens and the MLP polishes eac'
  [06:12 – 06:20] 'At each step, attention mi

---
## 8b. Unified Translation Mode — VLM frames + Whisper Turbo + BGE

A second video pipeline that complements the InternVideo2 shared-space approach
above. Here we **translate every modality into text** before retrieval:

```
MP4 ──► strategic frame sampling (1 every k secs, dHash novelty filter)
                                                     │
                                                     ▼
                                            Qwen2.5-VL (VLM) ──► per-frame summaries
        ffmpeg audio ──► faster-whisper Turbo CT2 ──► timestamped transcript

                  ┌──── smart timestamp alignment ────┐
                  │  vision-driven time windows:      │
                  │  boundaries = kept frame times    │
                  │  transcript ∩ window → speech     │
                  └────────────────┬──────────────────┘
                                   ▼
                    UnifiedSegment (vision + transcript)
                                   │
                                   ▼
                BGE-m3 embeddings ──► Qdrant unified collection
                                   │
                text query ────────►│ similarity search + BM25 + reranker
                                   ▼
                            top-k segments ──► LLM (text only)
```

**Why dual pipelines?** IV2 retrieves directly on visual content (good for "show me
the diagram with the residual connection"); the unified text pipeline benefits
from the LLM's familiarity with text and from BGE's strong retrieval baseline
(good for "how are attention weights computed?"). Section 9 below compares both on the
same questions.

**Design choices spelled out:**
- *k = 30 s default* — short videos with talking-head/slide format barely change
  on a second scale; sparser sampling avoids redundant VLM calls.
- *dHash 8×8, Hamming ≥ 8* — drops near-duplicate frames (e.g. the same slide
  shown for 2 minutes); cheap, no extra model dependency, more discriminating
  than a histogram for slide/diagram content.
- *Vision-driven time windows* — boundaries snap to retained frames rather than
  fixed-length blocks, so transcript chunks track real visual shifts (a long
  static slide produces one large window with all its speech).

**Models loaded here are additive** — the IV2, HF-Whisper, generator and
cross-encoder loaders above are unchanged.


### 8b.1 Unified Mode Configuration


In [ ]:
# Unified Translation Mode — env-driven knobs.
# Loaded *after* the original config so it inherits VIDEO_PATH, PERSIST_DIR,
# RETRIEVER_K, RERANKER_TOP_N, ENABLE_RERANKING from Section 1.

# ── Strategic frame sampling ──────────────────────────────────────────────────
# Sample one frame every K seconds. Short videos (<10 min) default to k=30:
# enough density to catch slide/scene changes without burning VLM calls on
# near-identical talking-head frames.
UNIFIED_SAMPLE_EVERY_SECS = int(os.getenv("UNIFIED_SAMPLE_EVERY_SECS", "30"))

# dHash size and Hamming threshold for the novelty filter.
# 8×8 → 64-bit fingerprint. Threshold = 8 means "differ in >=8 of 64 bits".
# Lowering it keeps fewer frames; raising it keeps more.
UNIFIED_DHASH_SIZE = int(os.getenv("UNIFIED_DHASH_SIZE", "8"))
UNIFIED_NOVELTY_HAMMING = int(os.getenv("UNIFIED_NOVELTY_HAMMING", "8"))

# ── Whisper Turbo (faster-whisper CT2) — separate loader, additive ────────────
# `WHISPER_MODEL` in Section 1 stays on HF whisper-large-v3 (used by IV2 mode
# for LLM-context transcript). Unified mode uses Turbo via faster-whisper.
WHISPER_TURBO_MODEL = os.getenv(
    "WHISPER_TURBO_MODEL", "deepdml/faster-whisper-large-v3-turbo-ct2"
)
WHISPER_TURBO_DEVICE       = os.getenv("WHISPER_TURBO_DEVICE", "auto").lower()
WHISPER_TURBO_COMPUTE_TYPE = os.getenv("WHISPER_TURBO_COMPUTE_TYPE", "float16")
# Keep the default conservative: the IV2 model is still resident when Turbo runs.
WHISPER_TURBO_BATCH_SIZE   = int(os.getenv("WHISPER_TURBO_BATCH_SIZE", "4"))
WHISPER_TURBO_BEAM_SIZE    = int(os.getenv("WHISPER_TURBO_BEAM_SIZE", "1"))
WHISPER_TURBO_VAD          = os.getenv("WHISPER_TURBO_VAD", "true").lower() == "true"

# ── VLM (Qwen2.5-VL) — same model class as rag_pipeline_imgs ─────────────────
VLM_MODEL              = os.getenv("VLM_MODEL", "Qwen/Qwen2.5-VL-3B-Instruct")
VLM_MAX_NEW_TOKENS     = int(os.getenv("VLM_MAX_NEW_TOKENS", "300"))

# ── BGE-m3 — same text embedder as rag_pipeline_imgs and rag_pipeline_audio ──
BGE_MODEL              = os.getenv("BGE_MODEL", "BAAI/bge-m3")

# ── Unified Qdrant collection (separate from IV2 — different dim & semantics) ─
QDRANT_UNIFIED   = os.getenv("QDRANT_UNIFIED",   f"video_unified_{VIDEO_SOURCE_ID}")
RESET_UNIFIED    = os.getenv("RESET_UNIFIED",    "false").lower() == "true"

# ── Persistence for unified mode caches ───────────────────────────────────────
UNIFIED_CACHE_DIR = Path(PERSIST_DIR) / "unified"
UNIFIED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Unified mode configuration:")
print(f"  Sample every       : {UNIFIED_SAMPLE_EVERY_SECS}s")
print(f"  dHash size/thresh  : {UNIFIED_DHASH_SIZE}×{UNIFIED_DHASH_SIZE} bits / Hamming ≥ {UNIFIED_NOVELTY_HAMMING}")
print(f"  Whisper Turbo      : {WHISPER_TURBO_MODEL}")
print(f"  VLM                : {VLM_MODEL}")
print(f"  BGE                : {BGE_MODEL}")
print(f"  Qdrant (unified)   : {QDRANT_UNIFIED}  reset={RESET_UNIFIED}")
print(f"  Cache dir          : {UNIFIED_CACHE_DIR}")


Unified mode configuration:
  Sample every       : 30s
  dHash size/thresh  : 8×8 bits / Hamming ≥ 8
  Whisper Turbo      : deepdml/faster-whisper-large-v3-turbo-ct2
  VLM                : Qwen/Qwen2.5-VL-3B-Instruct
  BGE                : BAAI/bge-m3
  Qdrant (unified)   : video_unified_467e468b2f83  reset=False
  Cache dir          : cache/iv2_only/unified


### 8b.2 Whisper Turbo Transcription (faster-whisper)

`deepdml/faster-whisper-large-v3-turbo-ct2` runs via CTranslate2. This cell
transcribes and caches the audio immediately, before Qwen-VL and BGE are loaded.
It then unloads Whisper Turbo to keep GPU memory pressure low. Turbo is roughly
8x faster than large-v3 while retaining high accuracy.


In [ ]:
import gc


def _preload_ctranslate2_cuda_libs() -> None:
    """
    Preload pip-installed cuBLAS/cuDNN libraries for CTranslate2 in WSL.

    VS Code/Jupyter kernels do not always inherit LD_LIBRARY_PATH from an
    interactive shell. Without this preload, faster-whisper can terminate the
    kernel natively with a missing libcudnn_ops.so.9 error.
    """
    import ctypes
    import nvidia.cublas.lib
    import nvidia.cudnn.lib

    cublas_dir = Path(nvidia.cublas.lib.__file__).parent
    cudnn_dir = Path(nvidia.cudnn.lib.__file__).parent
    current = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(
        str(path) for path in [cublas_dir, cudnn_dir] if str(path)
    ) + (f":{current}" if current else "")

    libraries = [
        cublas_dir / "libcublasLt.so.12",
        cublas_dir / "libcublas.so.12",
        cudnn_dir / "libcudnn.so.9",
        cudnn_dir / "libcudnn_ops.so.9",
        cudnn_dir / "libcudnn_graph.so.9",
        cudnn_dir / "libcudnn_engines_runtime_compiled.so.9",
        cudnn_dir / "libcudnn_engines_precompiled.so.9",
        cudnn_dir / "libcudnn_heuristic.so.9",
        cudnn_dir / "libcudnn_cnn.so.9",
        cudnn_dir / "libcudnn_adv.so.9",
    ]
    for library in libraries:
        ctypes.CDLL(str(library), mode=ctypes.RTLD_GLOBAL)


WHISPER_TURBO_AVAILABLE = False
whisper_turbo_model     = None
whisper_turbo_batched   = None

_turbo_cache = UNIFIED_CACHE_DIR / f"transcript_turbo_{_file_hash(VIDEO_PATH)}.json"


def _load_cached_turbo_transcript() -> Optional[List[Dict]]:
    if not _turbo_cache.exists():
        return None
    print(f"✓ Whisper Turbo transcript cached: {_turbo_cache.name}")
    return json.loads(_turbo_cache.read_text())


def _transcribe_whisper_turbo_now(audio_path: str) -> List[Dict]:
    """
    Transcribe before loading Qwen-VL and BGE.

    CTranslate2 inference starts in this cell while GPU memory pressure is low.
    The transcript is persisted immediately, then the Turbo model is unloaded.
    """
    if not WHISPER_TURBO_AVAILABLE or whisper_turbo_batched is None:
        raise RuntimeError("Whisper Turbo not loaded.")

    print("Running Whisper Turbo transcription before loading VLM/BGE ...")
    seg_iter, info = whisper_turbo_batched.transcribe(
        audio_path,
        batch_size      = WHISPER_TURBO_BATCH_SIZE,
        language        = WHISPER_LANGUAGE or None,
        vad_filter      = WHISPER_TURBO_VAD,
        beam_size       = WHISPER_TURBO_BEAM_SIZE,
        word_timestamps = False,
    )
    segs: List[Dict] = []
    for segment in seg_iter:
        text = segment.text.strip()
        if text:
            segs.append({
                "text": text,
                "start": float(segment.start),
                "end": float(segment.end),
            })

    if hasattr(info, "language") and info.language:
        print(f"  Detected: {info.language} (p={info.language_probability:.2f})")
    print(f"  ✓ {len(segs)} Whisper-Turbo segments.")
    _turbo_cache.write_text(json.dumps(segs, ensure_ascii=False, indent=2))
    return segs


turbo_segments = _load_cached_turbo_transcript()
if turbo_segments is None:
    print(f"Loading Whisper Turbo: {WHISPER_TURBO_MODEL} ...")
    try:
        from faster_whisper import WhisperModel, BatchedInferencePipeline

        if WHISPER_TURBO_DEVICE not in {"auto", "cuda", "cpu"}:
            raise ValueError(
                "WHISPER_TURBO_DEVICE must be one of: auto, cuda, cpu"
            )
        _wt_device = (
            "cuda" if WHISPER_TURBO_DEVICE == "auto" and torch.cuda.is_available()
            else "cpu" if WHISPER_TURBO_DEVICE == "auto"
            else WHISPER_TURBO_DEVICE
        )
        _wt_compute_type = (
            WHISPER_TURBO_COMPUTE_TYPE if _wt_device == "cuda" else "int8"
        )
        if _wt_device == "cuda":
            _preload_ctranslate2_cuda_libs()

        whisper_turbo_model = WhisperModel(
            WHISPER_TURBO_MODEL,
            device       = _wt_device,
            compute_type = _wt_compute_type,
        )
        whisper_turbo_batched   = BatchedInferencePipeline(model=whisper_turbo_model)
        WHISPER_TURBO_AVAILABLE = True
        print(
            f"  ✓ Whisper Turbo ready | device={_wt_device} | "
            f"compute={_wt_compute_type} | batch={WHISPER_TURBO_BATCH_SIZE}"
        )
        turbo_segments = _transcribe_whisper_turbo_now(str(audio_track))
    except Exception as e:
        print(f"  ✗ faster-whisper failed: {e}")
        print("    Install with: pip install faster-whisper")
        print("    For a low-memory fallback set WHISPER_TURBO_DEVICE=cpu in setup.env.")
        raise
    finally:
        # The transcript is materialized. Free CT2 GPU memory before Qwen-VL/BGE load.
        whisper_turbo_batched   = None
        whisper_turbo_model     = None
        WHISPER_TURBO_AVAILABLE = False
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("  ✓ Whisper Turbo unloaded after transcription.")


Loading Whisper Turbo: deepdml/faster-whisper-large-v3-turbo-ct2 ...
  ✓ Whisper Turbo ready | device=cuda | compute=float16 | batch=4
Running Whisper Turbo transcription before loading VLM/BGE ...
  Detected: en (p=1.00)
  ✓ 21 Whisper-Turbo segments.
  ✓ Whisper Turbo unloaded after transcription.


### 8b.3 VLM Loader (Qwen2.5-VL)

Same loader pattern as `rag_pipeline_imgs` Section 3: dispatch on the model
name to pick the right `Qwen…ForConditionalGeneration` class. Falls back
gracefully if the model fails to load — frame summarisation will skip and
unified segments will fall back to transcript-only content.


In [ ]:
QWEN_VL_AVAILABLE   = False
qwen_vl_model       = None
qwen_vl_processor   = None
_process_vision_info = None

print(f"Loading VLM: {VLM_MODEL} ...")

try:
    from transformers import AutoProcessor

    if "qwen2.5-vl" in VLM_MODEL.lower():
        from transformers import Qwen2_5_VLForConditionalGeneration as _QwenVLClass
    elif "qwen2-vl" in VLM_MODEL.lower():
        from transformers import Qwen2VLForConditionalGeneration as _QwenVLClass
    else:
        raise ValueError(
            f"VLM_MODEL={VLM_MODEL!r} is not a recognised Qwen-VL checkpoint. "
            "Use a 'Qwen/Qwen2.5-VL-*' or 'Qwen/Qwen2-VL-*' model."
        )

    from qwen_vl_utils import process_vision_info as _process_vision_info

    qwen_vl_model = _QwenVLClass.from_pretrained(
        VLM_MODEL,
        torch_dtype       = "auto",
        device_map        = "auto",
        trust_remote_code = True,
    )
    qwen_vl_processor = AutoProcessor.from_pretrained(
        VLM_MODEL, trust_remote_code=True
    )
    QWEN_VL_AVAILABLE = True
    print(f"  ✓ VLM ready: {VLM_MODEL}")

except Exception as e:
    print(f"  ✗ Could not load {VLM_MODEL}: {e}")
    print("    Frame summarisation will be skipped; unified segments will rely")
    print("    on transcript only.")


Loading VLM: Qwen/Qwen2.5-VL-3B-Instruct ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


  ✓ VLM ready: Qwen/Qwen2.5-VL-3B-Instruct


### 8b.4 BGE-m3 Loader

The text embedder for the unified pipeline. Cosine similarity, normalised
outputs — identical config to the other two notebooks for cross-pipeline
comparability.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

BGE_AVAILABLE  = False
bge_embeddings = None
BGE_DIM        = 1024   # bge-m3 default; overwritten by probe below

print(f"Loading BGE: {BGE_MODEL} ...")

try:
    bge_embeddings = HuggingFaceEmbeddings(
        model_name    = BGE_MODEL,
        model_kwargs  = {"trust_remote_code": True},
        encode_kwargs = {"normalize_embeddings": True},
    )
    BGE_DIM       = len(bge_embeddings.embed_query("dim probe"))
    BGE_AVAILABLE = True
    print(f"  ✓ BGE ready (dim={BGE_DIM})")
except Exception as e:
    print(f"  ✗ BGE: {e}")


Loading BGE: BAAI/bge-m3 ...
  ✓ BGE ready (dim=1024)


### 8b.5 Strategic Frame Sampling with dHash Novelty Filter

Walk the video at one frame every `UNIFIED_SAMPLE_EVERY_SECS` seconds. For
each candidate, compute a difference-hash (dHash) and compare against the
hashes of all previously kept frames. Skip the frame unless its minimum
Hamming distance is at least `UNIFIED_NOVELTY_HAMMING` bits.

> *Why dHash?* It's a 1-page algorithm (greyscale → resize → adjacent-pixel
> comparison), zero extra dependency, and very effective at separating
> different slides while collapsing near-duplicates. For talking-head shots
> with a static background this prunes ~80–90% of candidates.


In [ ]:
from PIL import Image as PILImage

@dataclass
class SampledFrame:
    """A frame retained by the novelty filter."""
    timestamp_sec: float
    image:         object               # PIL Image
    dhash:         np.ndarray           # boolean array of size DHASH_SIZE^2
    frame_idx:     int    = 0

def _dhash(img, size: int = UNIFIED_DHASH_SIZE) -> np.ndarray:
    """
    Difference hash — boolean array of length size*size.

    Algorithm: greyscale, resize to (size+1, size), compare each pixel to
    its right neighbour. Robust to brightness, JPEG noise, and small shifts.
    """
    g = img.convert("L").resize((size + 1, size), PILImage.BILINEAR)
    a = np.asarray(g, dtype=np.int32)
    return (a[:, 1:] > a[:, :-1]).flatten()

def _hamming(a: np.ndarray, b: np.ndarray) -> int:
    return int(np.sum(a != b))

def sample_frames_strategic(
    video_path:        str,
    every_secs:        int = UNIFIED_SAMPLE_EVERY_SECS,
    novelty_threshold: int = UNIFIED_NOVELTY_HAMMING,
    verbose:           bool = True,
) -> List[SampledFrame]:
    """
    Sample one frame per `every_secs`, keep only those whose dHash differs
    by at least `novelty_threshold` Hamming bits from every previously kept frame.
    """
    vr        = decord.VideoReader(video_path, ctx=decord.cpu(0))
    fps       = vr.get_avg_fps()
    total     = len(vr)
    duration  = total / fps

    if verbose:
        print(f"Video: {duration:.1f}s | {total} frames @ {fps:.2f} fps")
        print(f"Stride: 1 frame every {every_secs}s | novelty ≥ {novelty_threshold} bits")

    kept:    List[SampledFrame] = []
    skipped: int                 = 0
    t:       float               = 0.0

    while t < duration:
        idx = min(int(t * fps), total - 1)
        arr = vr[idx].asnumpy()
        img = PILImage.fromarray(arr)
        h   = _dhash(img)

        # Novelty test against ALL kept frames (not just the previous one).
        # This catches the case where the video periodically returns to an
        # earlier slide — we don't re-summarise it.
        is_novel = all(_hamming(h, k.dhash) >= novelty_threshold for k in kept)

        if is_novel:
            kept.append(SampledFrame(
                timestamp_sec = t,
                image         = img,
                dhash         = h,
                frame_idx     = idx,
            ))
            if verbose:
                print(f"  [{t:7.1f}s] ✓ kept   (#{len(kept):>2d})")
        else:
            skipped += 1
            if verbose:
                print(f"  [{t:7.1f}s] · skip   (near-duplicate)")

        t += every_secs

    if verbose:
        print(f"\nSampling complete: {len(kept)} kept, {skipped} skipped.")
    return kept

# Run the sampler
sampled_frames = sample_frames_strategic(VIDEO_PATH)


Video: 603.1s | 14459 frames @ 23.98 fps
Stride: 1 frame every 30s | novelty ≥ 8 bits
  [    0.0s] ✓ kept   (# 1)
  [   30.0s] ✓ kept   (# 2)
  [   60.0s] ✓ kept   (# 3)
  [   90.0s] ✓ kept   (# 4)
  [  120.0s] ✓ kept   (# 5)
  [  150.0s] ✓ kept   (# 6)
  [  180.0s] · skip   (near-duplicate)
  [  210.0s] ✓ kept   (# 7)
  [  240.0s] ✓ kept   (# 8)
  [  270.0s] ✓ kept   (# 9)
  [  300.0s] ✓ kept   (#10)
  [  330.0s] · skip   (near-duplicate)
  [  360.0s] · skip   (near-duplicate)
  [  390.0s] ✓ kept   (#11)
  [  420.0s] ✓ kept   (#12)
  [  450.0s] ✓ kept   (#13)
  [  480.0s] · skip   (near-duplicate)
  [  510.0s] ✓ kept   (#14)
  [  540.0s] ✓ kept   (#15)
  [  570.0s] ✓ kept   (#16)
  [  600.0s] ✓ kept   (#17)

Sampling complete: 17 kept, 4 skipped.


### 8b.6 VLM Frame Summarisation

For each retained frame, call Qwen2.5-VL to produce a 2–4 sentence description.
Summaries are cached per-video so re-running the cell is cheap.


In [ ]:
import io, base64

# ── Cache: video-hash → list of summaries aligned with sampled_frames ─────────
_vlm_cache_path = UNIFIED_CACHE_DIR / f"vlm_summaries_{_file_hash(VIDEO_PATH)}_{VLM_MODEL.split('/')[-1]}.json"

def summarize_frame_vlm(img, max_new_tokens: int = VLM_MAX_NEW_TOKENS) -> str:
    """Single-frame VLM summary. Falls back to a placeholder if VLM not loaded."""
    if not QWEN_VL_AVAILABLE:
        return "[VLM unavailable]"

    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"data:image/jpeg;base64,{b64}"},
            {"type": "text",
             "text":  "Describe what is shown in this video frame in 2–4 sentences. "
                      "Be specific about visible text, diagrams, charts, slides, or scenes. "
                      "Do not hallucinate details that are not visible."},
        ],
    }]

    text_in = qwen_vl_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    img_inputs, vid_inputs = _process_vision_info(messages)
    inputs = qwen_vl_processor(
        text   = [text_in],
        images = img_inputs,
        videos = vid_inputs,
        padding        = True,
        return_tensors = "pt",
    ).to(qwen_vl_model.device)

    out     = qwen_vl_model.generate(**inputs, max_new_tokens=max_new_tokens)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    return qwen_vl_processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()


def summarize_frames_cached(frames: List[SampledFrame]) -> List[str]:
    """Vectorised wrapper with on-disk cache keyed by (video_hash, model)."""
    # Load cache: list aligned by timestamp_sec rounded to 1 decimal
    cache: Dict[str, str] = {}
    if _vlm_cache_path.exists():
        try:
            cache = json.loads(_vlm_cache_path.read_text())
        except Exception as e:
            print(f"  [warn] could not read VLM cache: {e}")

    summaries: List[str] = []
    new_cache_entries: Dict[str, str] = dict(cache)

    for i, f in enumerate(frames):
        key = f"{f.timestamp_sec:.1f}"
        if key in cache:
            summaries.append(cache[key])
            print(f"  [{i+1}/{len(frames)}] {f.timestamp_sec:6.1f}s  ✓ cached")
            continue
        try:
            s = summarize_frame_vlm(f.image)
            summaries.append(s)
            new_cache_entries[key] = s
            print(f"  [{i+1}/{len(frames)}] {f.timestamp_sec:6.1f}s  ✓ {s[:80]!r}")
        except Exception as e:
            print(f"  [{i+1}/{len(frames)}] {f.timestamp_sec:6.1f}s  ✗ {e}")
            summaries.append("[summarisation error]")

    # Persist
    try:
        _vlm_cache_path.write_text(json.dumps(new_cache_entries, ensure_ascii=False, indent=2))
    except Exception as e:
        print(f"  [warn] could not write VLM cache: {e}")

    return summaries

vlm_frame_summaries = summarize_frames_cached(sampled_frames)
print(f"\n✓ {sum(1 for s in vlm_frame_summaries if not s.startswith('['))}/{len(sampled_frames)} frames summarised.")


  [1/17]    0.0s  ✓ 'The video frame shows a person wearing a dark hoodie with a blue rectangular bor'
  [2/17]   30.0s  ✓ 'The image shows a slide with the title "Attention Is All You Need Explained." On'
  [3/17]   60.0s  ✓ 'The image shows a diagram from a presentation titled "Attention Is All You Need '
  [4/17]   90.0s  ✓ 'The image shows a slide from a presentation titled "Attention Is All You Need Ex'
  [5/17]  120.0s  ✓ 'The video frame displays the logo and name "clerk" on a dark background. The log'
  [6/17]  150.0s  ✓ 'The image shows a slide from a presentation titled "Attention Is All You Need Ex'
  [7/17]  210.0s  ✓ 'The image shows a diagram from a presentation titled "Attention Is All You Need '
  [8/17]  240.0s  ✓ 'The image shows a slide from a presentation titled "Attention Is All You Need Ex'
  [9/17]  270.0s  ✓ 'The image shows a slide from a presentation titled "Attention Is All You Need Ex'
  [10/17]  300.0s  ✓ 'The image shows a diagram from a presentation title

### 8b.7 Whisper Turbo + Smart Timestamp Alignment

Two responsibilities:

1. **Transcribe** the extracted audio with Whisper Turbo, keeping segment-level
   `(start, end)` timestamps. Output is cached per video hash.
2. **Build vision-aligned time windows**: the kept-frame timestamps act as
   boundaries, and each window's transcript is the union of Whisper segments
   that overlap it. A long static scene (one retained frame) yields one wide
   window with all its speech; a busy scene-change-heavy stretch yields many
   short windows. The last window extends to the end of the video.


In [ ]:
# Whisper Turbo ran earlier, before Qwen-VL and BGE were loaded.
# This cell only consumes the persisted transcript and aligns it to frame windows.
if "turbo_segments" not in globals():
    _turbo_cache = UNIFIED_CACHE_DIR / f"transcript_turbo_{_file_hash(VIDEO_PATH)}.json"
    if not _turbo_cache.exists():
        raise RuntimeError(
            "Whisper Turbo transcript missing. Run Section 8b.2 before loading VLM/BGE."
        )
    turbo_segments = json.loads(_turbo_cache.read_text())
    print(f"✓ Whisper Turbo transcript cached: {_turbo_cache.name}")

# ── Smart timestamp alignment ────────────────────────────────────────────────
@dataclass
class UnifiedSegment:
    """One vision-aligned time window. Vision summary + matching speech."""
    start_sec:       float
    end_sec:         float
    vision_summary:  str   = ""
    transcript:      str   = ""
    source_file:     str   = ""
    frame_idx:       int   = 0

    @property
    def timestamp_label(self) -> str:
        def fmt(s): m, sec = divmod(int(s), 60); return f"{m:02d}:{sec:02d}"
        return f"[{fmt(self.start_sec)} – {fmt(self.end_sec)}]"

    @property
    def content(self) -> str:
        # Single text view for embedding, BM25, retrieval-precision and
        # context-recall metrics. Vision summary first so an image-driven
        # query has its strongest hits up top of the embedding window.
        return f"VISION: {self.vision_summary.strip()}\nTRANSCRIPT: {self.transcript.strip()}".strip()

    @property
    def doc_type(self) -> str:
        return "unified"


def build_unified_segments(
    frames:          List[SampledFrame],
    vlm_summaries:   List[str],
    turbo_segs:      List[Dict],
    video_duration:  float,
    source_file:     str,
) -> List[UnifiedSegment]:
    """
    Window boundaries = retained-frame timestamps + video_duration.
    Each window i covers [frames[i].timestamp_sec, boundaries[i+1]).

    Transcript for window i is the concatenation (in time order) of all
    Whisper segments that overlap the window, with no padding — overlaps
    are handled at the boundary because Whisper segments rarely split
    exactly on visual cuts.
    """
    if not frames:
        return []

    boundaries = [f.timestamp_sec for f in frames] + [video_duration]
    unified: List[UnifiedSegment] = []

    for i, f in enumerate(frames):
        start, end = boundaries[i], boundaries[i + 1]

        # Whisper segments whose [s.start, s.end] intersects [start, end].
        overlap = [s for s in turbo_segs if s["end"] >= start and s["start"] <= end]
        transcript = " ".join(s["text"] for s in overlap).strip()

        unified.append(UnifiedSegment(
            start_sec      = start,
            end_sec        = end,
            vision_summary = vlm_summaries[i] if i < len(vlm_summaries) else "",
            transcript     = transcript,
            source_file    = source_file,
            frame_idx      = f.frame_idx,
        ))

    return unified

unified_segments = build_unified_segments(
    sampled_frames, vlm_frame_summaries, turbo_segments,
    _video_duration if "_video_duration" in dir() else (len(decord.VideoReader(VIDEO_PATH, ctx=decord.cpu(0))) / decord.VideoReader(VIDEO_PATH, ctx=decord.cpu(0)).get_avg_fps()),
    VIDEO_PATH,
)
print(f"\nBuilt {len(unified_segments)} unified segments.")
for u in unified_segments[:3]:
    print(f"  {u.timestamp_label}")
    print(f"    vision : {u.vision_summary[:90]!r}")
    print(f"    speech : {u.transcript[:90]!r}")



Built 17 unified segments.
  [00:00 – 00:30]
    vision : 'The video frame shows a person wearing a dark hoodie with a blue rectangular border around'
    speech : 'How did a single paper attention is all you need reshape the entire AI landscape? In this '
  [00:30 – 01:00]
    vision : 'The image shows a slide with the title "Attention Is All You Need Explained." On the left '
    speech : 'An ML model maps features like the number of bedrooms, location, and zip code to a price. '
  [01:00 – 01:30]
    vision : 'The image shows a diagram from a presentation titled "Attention Is All You Need Explained.'
    speech : 'For example, a linear layer applies a linear transformation to its input. By stacking seve'


### 8b.8 BGE Embedding + Qdrant Indexing

Embed each segment's combined `VISION: … / TRANSCRIPT: …` string with BGE-m3
and upsert into a dedicated Qdrant collection. Separate from the source-scoped InternVideo2 collection
because the vector dimensionality and semantics differ (BGE 1024-dim text
embeddings vs IV2 CLIP-style 512-dim video-text embeddings).


In [ ]:
# ── Unified Qdrant collection ────────────────────────────────────────────────
if RESET_UNIFIED and client.collection_exists(QDRANT_UNIFIED):
    client.delete_collection(QDRANT_UNIFIED)
    print(f"✓ Deleted '{QDRANT_UNIFIED}'")

if not client.collection_exists(QDRANT_UNIFIED):
    client.create_collection(
        collection_name = QDRANT_UNIFIED,
        vectors_config  = VectorParams(size=BGE_DIM, distance=Distance.COSINE),
    )
    print(f"✓ Created '{QDRANT_UNIFIED}' (dim={BGE_DIM})")
else:
    print(f"✓ Using existing '{QDRANT_UNIFIED}'")

unified_docstore: Dict[str, UnifiedSegment] = {}

def index_unified_segments(segs: List[UnifiedSegment]) -> List[str]:
    if not BGE_AVAILABLE:
        print("BGE not available — skipping unified index.")
        return []
    if not segs:
        return []

    # Skip empty segments (no vision AND no transcript)
    valid = [s for s in segs if s.content.strip()
                                  and s.content.strip() not in ("VISION:", "TRANSCRIPT:")]
    if len(valid) < len(segs):
        print(f"  Skipping {len(segs) - len(valid)} empty segments.")

    texts = [s.content for s in valid]
    print(f"Embedding {len(texts)} unified segments with BGE...")
    vecs  = bge_embeddings.embed_documents(texts)

    points: List[PointStruct] = []
    ids:    List[str]         = []
    for seg, vec in zip(valid, vecs):
        uid = str(uuid.uuid5(
            uuid.NAMESPACE_URL,
            f"{VIDEO_SOURCE_ID}:unified:{seg.start_sec:.3f}:{seg.end_sec:.3f}",
        ))
        unified_docstore[uid] = seg
        ids.append(uid)
        points.append(PointStruct(
            id      = uid,
            vector  = vec,
            payload = {
                "start_sec":          float(seg.start_sec),
                "end_sec":            float(seg.end_sec),
                "timestamp":          seg.timestamp_label,
                "vision_preview":     seg.vision_summary[:160],
                "transcript_preview": seg.transcript[:160],
                "vision_summary":     seg.vision_summary,
                "transcript":         seg.transcript,
                "source_file":        str(seg.source_file),
            },
        ))

    for i in range(0, len(points), 100):
        client.upsert(collection_name=QDRANT_UNIFIED, points=points[i:i+100])

    print(f"✓ Indexed {len(ids)} unified segments into '{QDRANT_UNIFIED}'.")
    return ids

unified_ids = index_unified_segments(unified_segments)

# ── BM25 over unified segment content (for hybrid retrieval) ──────────────────
from rank_bm25 import BM25Okapi

class UnifiedBM25:
    def __init__(self):
        self._segs: List[UnifiedSegment] = []
        self._bm25 = None

    def build(self, segs: List[UnifiedSegment]):
        self._segs = [s for s in segs if s.content.strip()]
        tokenized  = [s.content.lower().split() for s in self._segs]
        self._bm25 = BM25Okapi(tokenized) if tokenized else None
        print(f"BM25 built on {len(self._segs)} unified segments.")

    def retrieve(self, query: str, top_k: int = RETRIEVER_K):
        if self._bm25 is None: return []
        scores  = self._bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(float(scores[i]), self._segs[i]) for i in top_idx if scores[i] > 0]

unified_bm25 = UnifiedBM25()
unified_bm25.build(unified_segments)


✓ Using existing 'video_unified_467e468b2f83'
Embedding 17 unified segments with BGE...
✓ Indexed 17 unified segments into 'video_unified_467e468b2f83'.
BM25 built on 17 unified segments.


### 8b.9 Unified Retrieval & Generation

Hybrid retrieval mirroring the audio pipeline:

1. **Dense:** BGE-encode query → cosine search in `QDRANT_UNIFIED`.
2. **BM25:** keyword search over combined `VISION + TRANSCRIPT` content.
3. **Dedupe + cross-encoder rerank** (when `ENABLE_RERANKING`).

Generation reuses the LLM from Section 7, but the prompt now exposes both the
visual description and the speech transcript for each retrieved window.


In [ ]:
def unified_retrieve(query: str) -> List[UnifiedSegment]:
    """BGE dense + BM25 + (optional) cross-encoder rerank over unified segments."""
    if not BGE_AVAILABLE:
        return []

    seen: set = set()
    cands: List[Tuple[str, UnifiedSegment]] = []

    # ── Dense ────────────────────────────────────────────────────────────────
    try:
        qvec = bge_embeddings.embed_query(query)
        hits = client.query_points(
            collection_name = QDRANT_UNIFIED,
            query           = qvec,
            limit           = RETRIEVER_K,
            with_payload    = True,
        ).points
        for h in hits:
            seg = unified_docstore.get(str(h.id))
            if seg and seg.content not in seen:
                seen.add(seg.content)
                cands.append((seg.content[:500], seg))
    except Exception as e:
        print(f"[unified_retrieve] dense error: {e}")

    # ── BM25 ─────────────────────────────────────────────────────────────────
    for _, seg in unified_bm25.retrieve(query, top_k=RETRIEVER_K):
        if seg.content not in seen:
            seen.add(seg.content)
            cands.append((seg.content[:500], seg))

    if not cands:
        return []

    # ── Rerank ───────────────────────────────────────────────────────────────
    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        pairs  = [(query, t) for t, _ in cands]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [c for _, c in cands]),
                        key=lambda x: x[0], reverse=True)
        return [c for _, c in ranked[:RERANKER_TOP_N]]
    return [c for _, c in cands[:RERANKER_TOP_N]]


def generate_answer_unified(segs: List[UnifiedSegment], question: str) -> str:
    """Re-uses the generator loaded in Section 7. Prompt exposes vision+speech per window."""
    if not GENERATOR_AVAILABLE:
        return "[Generator unavailable]"

    parts = []
    for s in segs:
        block = f"{s.timestamp_label}"
        if s.vision_summary.strip():
            block += f"\nVisual: {s.vision_summary.strip()}"
        if s.transcript.strip():
            block += f"\nSpeech: {s.transcript.strip()}"
        parts.append(block)
    context = "\n\n".join(parts) if parts else "[No context retrieved]"

    prompt = (
        "Answer the question using only the provided context excerpts. "
        "Each excerpt includes the video timestamp, a visual description of "
        "what is on-screen, and the speech transcript for that time window. "
        "If the context is insufficient, state that explicitly.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )

    if USE_OLLAMA:
        return _invoke_ollama_generator(prompt)

    messages = [{"role": "user", "content": prompt}]
    if hasattr(_gen_tok, "apply_chat_template"):
        text_in = _gen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text_in = prompt

    inputs = _gen_tok([text_in], return_tensors="pt").to(_gen_model.device)
    with torch.no_grad():
        out = _gen_model.generate(
            **inputs,
            max_new_tokens = GENERATION_MAX_NEW_TOKENS,
            do_sample      = False,
            temperature    = None,
            pad_token_id   = _gen_tok.eos_token_id,
        )
    return _gen_tok.batch_decode(
        out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0].strip()


# ── Demo ──────────────────────────────────────────────────────────────────────
_demo_q   = "How do queries, keys, and values update token representations in attention?"
_demo_docs = unified_retrieve(_demo_q)
_demo_ans  = generate_answer_unified(_demo_docs, _demo_q)
print("=" * 70)
print(f"Q: {_demo_q}\n\nA: {_demo_ans}\n\nSources:")
for s in _demo_docs:
    print(f"  {s.timestamp_label}")
    print(f"    vision : {s.vision_summary[:80]!r}")
    print(f"    speech : {s.transcript[:80]!r}")


Q: How do queries, keys, and values update token representations in attention?

A: Based on the provided context, here's how queries, keys, and values update token representations in attention:

1. **Queries**: These ask "what am I looking for?" For example, in the sentence "Jake learned AI even though it was difficult," the token 'it' forms a query vector implicitly asking what concept it is referring to.

2. **Keys**: These contain information about what each token has to offer. In the same sentence:
   - The key for 'Jake' describes that it holds information about a person.
   - The key for 'AI' describes that it represents a subject.

3. **Values**: These carry the actual content or meaning of the words. For instance, in the sentence:
   - The value for 'Jake' carries the meaning of being a person.
   - The value for 'AI' carries the meaning of being an artificial intelligence subject.

To update token representations:

- We take the **dot product** between a token's query and the 

---
## 9. Improved Evaluation

This section evaluates the video RAG approaches after both pipelines are available.

| Metric | What it checks |
|---|---|
| `BERTScore` | semantic similarity between generated and expected answer |
| `Precision` / `Recall` | lexical overlap of answer tokens against the expected answer |
| `Context Recall` | how many annotated claims are present in retrieved context |
| `Evidence Coverage` | source-aware coverage of claims in retrieved video evidence: visual claims are checked against frame/VLM summaries, speech/text claims against transcripts, and `both` claims against the combined evidence |
| `Must` / `Should` recall | answer claim coverage split by required vs desirable facts |

`Evidence Coverage` replaces plain timestamp coverage. For video, interval overlap alone is too weak: a segment can overlap the right time range while missing the visual or spoken fact. This metric checks whether the retrieved evidence actually supports the annotated claims.


### 9.1 Test Questions and Annotated Claims


In [ ]:
# These questions are grounded in the current educational video while
# remaining answerable by a Transformer-paper RAG baseline. Claims are
# tagged by evidence channel so Evidence Coverage checks the right source.
TEST_SET = [
    {
        "question": "Describe the overall architecture of the Transformer model",
        "expected_answer":
            "The Transformer uses an encoder-decoder architecture made of stacked "
            "blocks. Each block combines an attention layer, where tokens interact, "
            "with a feed-forward or MLP layer, where each token refines its own "
            "representation. Residual connections and layer normalization help keep "
            "training stable. The architecture also includes masked multi-head "
            "attention and cross-attention variants.",
        "claims": [
            {"text": "The Transformer includes an encoder and a decoder", "importance": "must", "source": "speech"},
            {"text": "The encoder and decoder are made of stacked blocks", "importance": "must", "source": "speech"},
            {"text": "Each block has an attention layer and a feed-forward or MLP layer", "importance": "must", "source": "speech"},
            {"text": "The attention layer lets tokens interact while the MLP layer refines each token representation", "importance": "must", "source": "speech"},
            {"text": "Residual connections and layer normalization keep training stable", "importance": "should", "source": "speech"},
            {"text": "The architecture diagram shows Multi-Head Attention, Masked Multi-Head Attention, Add and Norm, and Feed Forward layers", "importance": "should", "source": "visual", "start_sec": 210.0, "end_sec": 240.0},
        ],
        "requires_continuity": True,
        "requires_video_evidence": True,
        "evidence_windows": [
            {"label": "Architecture explanation", "start_sec": 201.42, "end_sec": 286.80, "timestamp": "[03:21 - 04:46]"},
            {"label": "Architecture diagram", "start_sec": 210.0, "end_sec": 240.0, "timestamp": "[03:30 - 04:00]"},
        ],
        "context_keywords": ["encoder", "decoder", "stacked blocks", "attention", "feedforward", "MLP", "residual", "layer normalization"],
        "answer_keywords": ["encoder", "decoder", "attention", "feed-forward", "MLP"],
    },
    {
        "question": "Why did Transformers improve on earlier RNN and LSTM sequence models?",
        "expected_answer":
            "Earlier RNNs and LSTMs process tokens one at a time while passing an "
            "internal memory forward. This sequential design prevents parallel "
            "processing and makes training slow. They also struggle with long-term "
            "dependencies because early information can be lost. Transformer attention "
            "lets tokens communicate directly, improving context handling and parallelism.",
        "claims": [
            {"text": "RNNs and LSTMs process one token at a time and pass an internal memory to the next step", "importance": "must", "source": "speech"},
            {"text": "Sequential processing prevents parallel processing and makes training slow", "importance": "must", "source": "speech"},
            {"text": "Earlier models struggle with long-term dependencies", "importance": "must", "source": "speech"},
            {"text": "Early information can be lost by the end of a long sequence", "importance": "should", "source": "speech"},
            {"text": "Attention lets all tokens in a sequence talk to each other directly", "importance": "must", "source": "speech"},
        ],
        "requires_continuity": True,
        "requires_video_evidence": True,
        "evidence_windows": [
            {"label": "RNN and LSTM limitations", "start_sec": 123.60, "end_sec": 173.42, "timestamp": "[02:03 - 02:53]"},
            {"label": "Direct token communication", "start_sec": 173.42, "end_sec": 201.36, "timestamp": "[02:53 - 03:21]"},
        ],
        "context_keywords": ["RNN", "LSTM", "sequential", "parallel", "long-term dependencies", "attention", "tokens"],
        "answer_keywords": ["sequential", "parallel", "long-term dependencies", "attention"],
    },
    {
        "question": "What different roles do the attention layer and the MLP layer play in a Transformer block?",
        "expected_answer":
            "The attention layer is the communication step: tokens look at other "
            "tokens and borrow information from the most relevant ones. The MLP or "
            "feed-forward layer is the private refinement step: each token updates "
            "its own representation individually. Repeating these two operations "
            "builds context-aware token representations.",
        "claims": [
            {"text": "The attention layer is where tokens interact", "importance": "must", "source": "speech"},
            {"text": "Tokens update themselves by looking at other tokens and borrowing relevant information", "importance": "must", "source": "speech"},
            {"text": "The MLP layer privately refines each token representation", "importance": "must", "source": "speech"},
            {"text": "Attention followed by MLP refinement builds contextual understanding", "importance": "should", "source": "speech"},
            {"text": "The slide contrasts an MLP process with an attention mechanism over the example sentence", "importance": "should", "source": "visual", "start_sec": 270.0, "end_sec": 300.0},
        ],
        "requires_continuity": True,
        "requires_video_evidence": True,
        "evidence_windows": [
            {"label": "Attention and MLP walkthrough", "start_sec": 201.42, "end_sec": 286.80, "timestamp": "[03:21 - 04:46]"},
            {"label": "MLP versus attention slide", "start_sec": 270.0, "end_sec": 300.0, "timestamp": "[04:30 - 05:00]"},
        ],
        "context_keywords": ["attention", "MLP", "interact", "refines", "representation", "contextual"],
        "answer_keywords": ["attention", "communication", "MLP", "refinement"],
    },
    {
        "question": "Why is positional information added to Transformer input embeddings?",
        "expected_answer":
            "A Transformer has no sense of token order by default, so positional "
            "information is added to input embeddings to tell the model where each "
            "token occurs in the sequence. Without it, sequences such as 'Jake learned "
            "AI' and 'AI learned Jake' could look the same to the model.",
        "claims": [
            {"text": "A tokenizer splits text into tokens and the tokens are embedded as numerical vectors", "importance": "should", "source": "speech"},
            {"text": "A Transformer has no sense of order by default", "importance": "must", "source": "speech"},
            {"text": "Positional information is added to embeddings to introduce token order", "importance": "must", "source": "speech"},
            {"text": "Without positional information Jake learned AI could look the same as AI learned Jake", "importance": "should", "source": "speech"},
            {"text": "The slide shows text tokenization and an embedding layer", "importance": "should", "source": "visual", "start_sec": 300.0, "end_sec": 330.0},
        ],
        "requires_continuity": False,
        "requires_video_evidence": True,
        "evidence_windows": [
            {"label": "Tokenization, embeddings, and position", "start_sec": 287.02, "end_sec": 323.06, "timestamp": "[04:47 - 05:23]"},
            {"label": "Tokenizer and embedding slide", "start_sec": 300.0, "end_sec": 330.0, "timestamp": "[05:00 - 05:30]"},
        ],
        "context_keywords": ["tokenizer", "tokens", "embedded", "order", "positional information", "embeddings"],
        "answer_keywords": ["positional", "order", "embeddings", "sequence"],
    },
    {
        "question": "How do queries, keys, and values determine which information a token receives in attention?",
        "expected_answer":
            "Each token produces a query, a key, and a value. The query represents "
            "what the token is looking for, a key represents what another token has "
            "to offer, and a value carries the content to share. Dot products between "
            "a query and the keys produce relevance scores. A softmax turns these into "
            "attention weights, and the token receives a weighted sum of the values.",
        "claims": [
            {"text": "The attention layer creates a query, a key, and a value for each token", "importance": "must", "source": "speech"},
            {"text": "A query asks what the token is looking for", "importance": "must", "source": "speech"},
            {"text": "A key describes what information a token has and a value carries the content to share", "importance": "must", "source": "speech"},
            {"text": "Dot products between a token query and other token keys produce relevance scores", "importance": "must", "source": "speech"},
            {"text": "A softmax normalizes scores into attention weights", "importance": "must", "source": "speech"},
            {"text": "The updated token representation is a weighted sum of value vectors", "importance": "must", "source": "both"},
            {"text": "The diagram shows query and key vectors, dot products, softmax, and weighted values", "importance": "should", "source": "visual", "start_sec": 390.0, "end_sec": 480.0},
        ],
        "requires_continuity": True,
        "requires_video_evidence": True,
        "evidence_windows": [
            {"label": "Query, key, and value roles", "start_sec": 344.66, "end_sec": 400.72, "timestamp": "[05:44 - 06:40]"},
            {"label": "Scores, softmax, and weighted values", "start_sec": 400.98, "end_sec": 458.59, "timestamp": "[06:40 - 07:38]"},
            {"label": "Attention diagrams", "start_sec": 390.0, "end_sec": 480.0, "timestamp": "[06:30 - 08:00]"},
        ],
        "context_keywords": ["query", "key", "value", "dot product", "softmax", "weights", "weighted sum"],
        "answer_keywords": ["query", "key", "value", "dot product", "softmax", "weighted sum"],
    },
    {
        "question": "How is attention computed efficiently for all tokens in parallel?",
        "expected_answer":
            "Instead of processing tokens one by one, the model stacks queries, keys, "
            "and values into matrices. It computes dot products and weighted sums "
            "simultaneously, allowing every token to communicate with every other token "
            "through parallel matrix operations. This is efficient and fully differentiable.",
        "claims": [
            {"text": "Queries keys and values are stacked into matrices", "importance": "must", "source": "speech"},
            {"text": "Dot products and weighted sums are performed simultaneously", "importance": "must", "source": "speech"},
            {"text": "Every token communicates with every other token in parallel matrix operations", "importance": "must", "source": "speech"},
            {"text": "The operation is efficient and fully differentiable", "importance": "should", "source": "speech"},
            {"text": "The slide shows Queries Keys and Values matrices with Wq Wk and Wv", "importance": "should", "source": "visual", "start_sec": 510.0, "end_sec": 540.0},
            {"text": "The diagram shows a softmax over query-key products followed by multiplication with values", "importance": "should", "source": "visual", "start_sec": 510.0, "end_sec": 540.0},
        ],
        "requires_continuity": False,
        "requires_video_evidence": True,
        "evidence_windows": [
            {"label": "Parallel matrix computation", "start_sec": 458.59, "end_sec": 515.86, "timestamp": "[07:38 - 08:35]"},
            {"label": "Matrix attention slide", "start_sec": 510.0, "end_sec": 540.0, "timestamp": "[08:30 - 09:00]"},
        ],
        "context_keywords": ["queries", "keys", "values", "matrices", "dot products", "weighted sums", "parallel"],
        "answer_keywords": ["matrices", "dot products", "weighted sums", "parallel"],
    },
    {
        "question": "Which attention variants does the video mention, and what broad purposes do they serve?",
        "expected_answer":
            "The video mentions masked attention, multi-head attention, and "
            "cross-attention. These variants modify the core attention calculation "
            "to help handle sequence order, enforce causality, and combine information "
            "from different sources. The visual diagram also shows scaled dot-product "
            "attention as a component inside the architecture.",
        "claims": [
            {"text": "The video mentions masked multi-head and cross-attention", "importance": "must", "source": "speech"},
            {"text": "Attention variants help handle sequence order", "importance": "must", "source": "speech"},
            {"text": "Attention variants help enforce causality", "importance": "must", "source": "speech"},
            {"text": "Attention variants help combine information from different sources", "importance": "must", "source": "speech"},
            {"text": "The diagram labels Multi-Head Attention, Scaled Dot-Product Attention, Masked Multi-Head Attention, and Cross attention", "importance": "should", "source": "visual", "start_sec": 540.0, "end_sec": 570.0},
            {"text": "Cross attention attends to source tokens", "importance": "should", "source": "visual", "start_sec": 540.0, "end_sec": 570.0},
        ],
        "requires_continuity": False,
        "requires_video_evidence": True,
        "evidence_windows": [
            {"label": "Attention variants explanation", "start_sec": 515.86, "end_sec": 545.74, "timestamp": "[08:35 - 09:05]"},
            {"label": "Attention variants architecture diagram", "start_sec": 540.0, "end_sec": 570.0, "timestamp": "[09:00 - 09:30]"},
        ],
        "context_keywords": ["masked", "multi-head", "cross-attention", "sequence order", "causality", "sources"],
        "answer_keywords": ["masked", "multi-head", "cross-attention", "causality", "sources"],
    },
]

TEST_SET_VIDEOS = TEST_SET

print(f"Evaluation test set: {len(TEST_SET)} video-grounded Transformer questions.")
print(f"  Video-evidence questions: {sum(1 for q in TEST_SET if q.get('requires_video_evidence', True))}")
print(f"  Must claims             : {sum(1 for q in TEST_SET for c in q['claims'] if c['importance'] == 'must')}")
print(f"  Should claims           : {sum(1 for q in TEST_SET for c in q['claims'] if c['importance'] == 'should')}")


Evaluation test set: 7 video-grounded Transformer questions.
  Video-evidence questions: 7
  Must claims             : 26
  Should claims           : 14


### 9.2 Retrieval Views and Answer Generation Wrappers


In [ ]:
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple


def _segment_key(seg) -> Tuple[float, float, str]:
    return (
        round(float(getattr(seg, "start_sec", 0.0)), 3),
        round(float(getattr(seg, "end_sec", 0.0)), 3),
        str(getattr(seg, "content", getattr(seg, "transcript", "")))[:120],
    )


def _dedupe_segments(segs: Iterable) -> List:
    seen = set()
    out = []
    for seg in segs:
        key = _segment_key(seg)
        if key in seen:
            continue
        seen.add(key)
        out.append(seg)
    return out


def _hit_id(hit) -> str:
    return str(getattr(hit, "id", ""))


def _hit_payload(hit) -> dict:
    payload = getattr(hit, "payload", None)
    return payload if isinstance(payload, dict) else {}


def _qdrant_dense_search(collection_name: str, query_vector, limit: int):
    result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=limit,
        with_payload=True,
    )
    return result.points


def _iv2_segment_from_hit(hit):
    seg = iv2_docstore.get(_hit_id(hit))
    if seg is not None:
        return seg
    payload = _hit_payload(hit)
    return VideoSegment(
        start_sec=float(payload.get("start_sec", 0.0)),
        end_sec=float(payload.get("end_sec", 0.0)),
        transcript=str(payload.get("transcript", payload.get("preview", ""))),
        source_file=str(payload.get("source_file", "")),
    )


def _unified_segment_from_hit(hit):
    seg = unified_docstore.get(_hit_id(hit))
    if seg is not None:
        return seg
    payload = _hit_payload(hit)
    return UnifiedSegment(
        start_sec=float(payload.get("start_sec", 0.0)),
        end_sec=float(payload.get("end_sec", 0.0)),
        vision_summary=str(payload.get("vision_summary", payload.get("vision_preview", ""))),
        transcript=str(payload.get("transcript", payload.get("transcript_preview", ""))),
        source_file=str(payload.get("source_file", "")),
    )


def retrieve_iv2_by_stage(query: str, retrieval_view: str = "rerank") -> List[VideoSegment]:
    """InternVideo2 supports dense search and optional cross-encoder rerank."""
    if not IV2_AVAILABLE:
        return []
    retrieval_view = retrieval_view or "rerank"
    if retrieval_view == "rerank":
        return iv2_retrieve(query)
    if retrieval_view != "dense":
        raise ValueError(f"Unsupported InternVideo2 retrieval view: {retrieval_view}")
    try:
        qvec = encode_text_iv2([query])[0].tolist()
        hits = _qdrant_dense_search(QDRANT_IV2_ONLY, qvec, RETRIEVER_K)
        return _dedupe_segments(_iv2_segment_from_hit(h) for h in hits)[:RERANKER_TOP_N]
    except Exception as e:
        print(f"[retrieve_iv2_by_stage] {e}")
        return []


def _retrieve_unified_dense(query: str, top_k: int = RETRIEVER_K) -> List[UnifiedSegment]:
    if not BGE_AVAILABLE:
        return []
    try:
        qvec = bge_embeddings.embed_query(query)
        hits = _qdrant_dense_search(QDRANT_UNIFIED, qvec, top_k)
        return _dedupe_segments(_unified_segment_from_hit(h) for h in hits)
    except Exception as e:
        print(f"[unified dense] {e}")
        return []


def _retrieve_unified_bm25(query: str, top_k: int = RETRIEVER_K) -> List[UnifiedSegment]:
    if "unified_bm25" not in globals() or unified_bm25 is None:
        return []
    try:
        return [seg for _, seg in unified_bm25.retrieve(query, top_k=top_k)]
    except Exception as e:
        print(f"[unified bm25] {e}")
        return []


def _rerank_segments(query: str, segs: Sequence, top_n: int = RERANKER_TOP_N) -> List:
    segs = list(segs)
    if not segs:
        return []
    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        pairs = [(query, str(getattr(seg, "content", getattr(seg, "transcript", "")))[:1000]) for seg in segs]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, segs), key=lambda x: x[0], reverse=True)
        return [seg for _, seg in ranked[:top_n]]
    return segs[:top_n]


def retrieve_unified_by_stage(query: str, retrieval_view: str = "rerank") -> List[UnifiedSegment]:
    """Unified VLM+Whisper+BGE supports dense, BM25, hybrid, and rerank views."""
    retrieval_view = retrieval_view or "rerank"
    if retrieval_view == "dense":
        return _retrieve_unified_dense(query)[:RERANKER_TOP_N]
    if retrieval_view == "bm25":
        return _retrieve_unified_bm25(query)[:RERANKER_TOP_N]
    if retrieval_view == "hybrid":
        return _dedupe_segments(_retrieve_unified_dense(query) + _retrieve_unified_bm25(query))[:RERANKER_TOP_N]
    if retrieval_view == "rerank":
        candidates = _dedupe_segments(_retrieve_unified_dense(query) + _retrieve_unified_bm25(query))
        return _rerank_segments(query, candidates, top_n=RERANKER_TOP_N)
    raise ValueError(f"Unsupported unified retrieval view: {retrieval_view}")


def generate_answer_for_approach(approach_key: str, segs: Sequence, question: str) -> str:
    if approach_key == "iv2":
        return generate_answer(list(segs), question)
    if approach_key == "unified":
        return generate_answer_unified(list(segs), question)
    raise ValueError(f"Unknown approach: {approach_key}")


VIDEO_EVAL_APPROACHES = {
    "iv2": {
        "label": "InternVideo2 shared video-text space",
        "retrieve": retrieve_iv2_by_stage,
        "views": {
            "dense": "Dense retrieval",
            "rerank": "Dense + cross-encoder rerank",
        },
    },
    "unified": {
        "label": "VLM frame summaries + Whisper + BGE",
        "retrieve": retrieve_unified_by_stage,
        "views": {
            "dense": "Dense retrieval",
            "bm25": "BM25 retrieval",
            "hybrid": "Dense + BM25 hybrid",
            "rerank": "Dense + BM25 + cross-encoder rerank",
        },
    },
}

print("Video evaluation approaches:")
for key, cfg in VIDEO_EVAL_APPROACHES.items():
    print(f"  - {cfg['label']}: {', '.join(cfg['views'].values())}")


Video evaluation approaches:
  - InternVideo2 shared video-text space: Dense retrieval, Dense + cross-encoder rerank
  - VLM frame summaries + Whisper + BGE: Dense retrieval, BM25 retrieval, Dense + BM25 hybrid, Dense + BM25 + cross-encoder rerank


### 9.3 Evaluation Metrics


In [ ]:
import math
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from bert_score import score as bert_score_score


EVAL_BERTSCORE_MODEL = os.getenv("EVAL_BERTSCORE_MODEL", "distilbert-base-uncased")
VISUAL_EVIDENCE_MIN_OVERLAP = float(os.getenv("VIDEO_VISUAL_EVIDENCE_MIN_OVERLAP", "0.30"))
_TOKEN_RE = re.compile(r"[a-z0-9]+")


def _tokens(text: str) -> List[str]:
    return _TOKEN_RE.findall(str(text).lower())


def precision_recall(prediction: str, reference: str) -> Tuple[float, float]:
    pred_tokens = _tokens(prediction)
    ref_tokens = _tokens(reference)
    if not pred_tokens or not ref_tokens:
        return 0.0, 0.0
    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)
    overlap = sum(min(pred_counts[t], ref_counts[t]) for t in pred_counts)
    return overlap / len(pred_tokens), overlap / len(ref_tokens)


def bertscore_f1(prediction: str, reference: str) -> float:
    if not str(prediction).strip() or str(prediction).startswith("[Generator unavailable]"):
        return np.nan
    _, _, f1 = bert_score_score(
        [str(prediction)],
        [str(reference)],
        lang="en",
        model_type=EVAL_BERTSCORE_MODEL,
        verbose=False,
        rescale_with_baseline=False,
    )
    return float(f1[0].item())


def _text_for_segment(seg, channel: str = "combined") -> str:
    if channel == "visual":
        return str(getattr(seg, "vision_summary", ""))
    if channel in {"speech", "text", "transcript"}:
        return str(getattr(seg, "transcript", ""))
    return str(getattr(seg, "content", getattr(seg, "transcript", "")))


def _joined_context(segs: Sequence, channel: str = "combined") -> str:
    return "\n\n".join(_text_for_segment(seg, channel) for seg in segs if _text_for_segment(seg, channel).strip())


def _soft_claim_match(claim_text: str, evidence_text: str, min_overlap: float = 0.50) -> bool:
    claim = str(claim_text).strip().lower()
    evidence = str(evidence_text).strip().lower()
    if not claim or not evidence:
        return False
    if claim in evidence:
        return True
    claim_tokens = [t for t in _tokens(claim) if len(t) > 2]
    evidence_tokens = set(_tokens(evidence))
    if not claim_tokens or not evidence_tokens:
        return False
    overlap = sum(1 for t in claim_tokens if t in evidence_tokens)
    return (overlap / len(claim_tokens)) >= min_overlap


def _claims_for(item: dict, importance: Optional[str] = None) -> List[dict]:
    claims = list(item.get("claims", []))
    if importance is not None:
        claims = [c for c in claims if c.get("importance") == importance]
    return claims


def claim_recall(text: str, claims: Sequence[dict]) -> Tuple[float, int, int]:
    claims = list(claims)
    if not claims:
        return np.nan, 0, 0
    covered = sum(1 for c in claims if _soft_claim_match(c.get("text", ""), text))
    return covered / len(claims), covered, len(claims)


def context_recall(segs: Sequence, item: dict) -> Tuple[float, int, int]:
    return claim_recall(_joined_context(segs, "combined"), _claims_for(item))


def _visual_window_retrieved(claim: dict, segs: Sequence) -> bool:
    """
    InternVideo2 retrieves raw clips rather than VLM summaries. For visual
    claims, count a retrieved clip when it overlaps the annotated on-screen
    evidence window. Unified VLM segments can still match semantically.
    """
    start = claim.get("start_sec")
    end = claim.get("end_sec")
    if start is None or end is None or float(end) <= float(start):
        return False
    target = (float(start), float(end))
    for seg in segs:
        seg_start = float(getattr(seg, "start_sec", 0.0))
        seg_end = float(getattr(seg, "end_sec", 0.0))
        if seg_end <= seg_start:
            continue
        overlap = max(0.0, min(target[1], seg_end) - max(target[0], seg_start))
        shorter = min(target[1] - target[0], seg_end - seg_start)
        if shorter > 0 and overlap / shorter >= VISUAL_EVIDENCE_MIN_OVERLAP:
            return True
    return False


def evidence_coverage(segs: Sequence, item: dict) -> Tuple[float, int, int]:
    """Source-aware semantic or raw-clip coverage for video evidence."""
    claims = _claims_for(item)
    if not claims:
        return np.nan, 0, 0

    visual_context = _joined_context(segs, "visual")
    speech_context = _joined_context(segs, "speech")
    combined_context = _joined_context(segs, "combined")

    covered = 0
    for claim in claims:
        source = str(claim.get("source", "both")).lower()
        text = claim.get("text", "")
        if source == "visual":
            ok = (
                _soft_claim_match(text, visual_context)
                or _visual_window_retrieved(claim, segs)
            )
        elif source in {"speech", "text", "transcript"}:
            ok = _soft_claim_match(text, speech_context)
        else:
            ok = (
                _soft_claim_match(text, combined_context)
                or _soft_claim_match(text, visual_context)
                or _soft_claim_match(text, speech_context)
            )
        covered += int(ok)
    return covered / len(claims), covered, len(claims)


def evaluate_video_answer(item: dict, answer: str, segs: Sequence, compute_bertscore: bool = True) -> dict:
    expected = item.get("expected_answer", "")
    precision, recall = precision_recall(answer, expected)
    ctx_recall, ctx_cov, ctx_total = context_recall(segs, item)
    ev_cov, ev_hits, ev_total = evidence_coverage(segs, item)
    must_recall, must_hits, must_total = claim_recall(answer, _claims_for(item, "must"))
    should_recall, should_hits, should_total = claim_recall(answer, _claims_for(item, "should"))

    return {
        "bertscore_f1": bertscore_f1(answer, expected) if compute_bertscore else np.nan,
        "precision": precision,
        "recall": recall,
        "context_recall": ctx_recall,
        "evidence_coverage": ev_cov,
        "must_recall": must_recall,
        "should_recall": should_recall,
        "context_claims_covered": f"{ctx_cov}/{ctx_total}",
        "evidence_claims_covered": f"{ev_hits}/{ev_total}",
        "must_covered": f"{must_hits}/{must_total}",
        "should_covered": f"{should_hits}/{should_total}",
        "n_retrieved": len(segs),
        "retrieved_visuals": sum(1 for s in segs if str(getattr(s, "vision_summary", "")).strip()),
    }


def summarize_video_eval(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    metric_cols = [
        "bertscore_f1",
        "precision",
        "recall",
        "context_recall",
        "evidence_coverage",
        "must_recall",
        "should_recall",
        "n_retrieved",
        "retrieved_visuals",
    ]
    summary = (
        df.groupby(["approach", "retrieval_view"], dropna=False)[metric_cols]
          .mean(numeric_only=True)
          .reset_index()
    )
    return summary.round(3)

print(f"BERTScore model: {EVAL_BERTSCORE_MODEL}")


BERTScore model: distilbert-base-uncased


### 9.4 Generate Evaluation Answers

Run this cell when you want to regenerate answers and metrics. The interactive table below only filters this cached dataframe, so changing filters is instant.


In [ ]:
EVAL_APPROACHES_TO_RUN = ["iv2", "unified"]
EVAL_STAGES_BY_APPROACH = {
    "iv2": ["dense", "rerank"],
    "unified": ["dense", "bm25", "hybrid", "rerank"],
}
EVAL_QUESTION_LIMIT = min(
    int(os.getenv("VIDEO_EVAL_QUESTION_LIMIT", str(len(TEST_SET)))),
    len(TEST_SET),
)
EVAL_COMPUTE_BERTSCORE = True


def build_video_eval_cache(
    question_limit: int = EVAL_QUESTION_LIMIT,
    approaches: Sequence[str] = EVAL_APPROACHES_TO_RUN,
    stages_by_approach: Dict[str, Sequence[str]] = EVAL_STAGES_BY_APPROACH,
    compute_bertscore: bool = EVAL_COMPUTE_BERTSCORE,
) -> pd.DataFrame:
    rows = []
    eval_items = TEST_SET[:question_limit]

    for approach_key in approaches:
        cfg = VIDEO_EVAL_APPROACHES[approach_key]
        for view_key in stages_by_approach.get(approach_key, cfg["views"].keys()):
            if view_key not in cfg["views"]:
                print(f"Skipping unsupported view {approach_key}/{view_key}")
                continue

            for q_idx, item in enumerate(eval_items, start=1):
                question = item["question"]
                print(f"[{cfg['label']} | {cfg['views'][view_key]}] Q{q_idx}: {question}")
                segs = cfg["retrieve"](question, view_key)
                answer = generate_answer_for_approach(approach_key, segs, question)
                metrics = evaluate_video_answer(item, answer, segs, compute_bertscore=compute_bertscore)

                rows.append({
                    "question_id": q_idx,
                    "approach_key": approach_key,
                    "approach": cfg["label"],
                    "retrieval_view_key": view_key,
                    "retrieval_view": cfg["views"][view_key],
                    "requires_video_evidence": bool(item.get("requires_video_evidence", True)),
                    "question": question,
                    "expected_answer": item.get("expected_answer", ""),
                    "answer": answer,
                    "sources": "\n".join(
                        f"{getattr(s, 'timestamp_label', '')} | "
                        f"visual={str(getattr(s, 'vision_summary', ''))[:90]!r} | "
                        f"speech={str(getattr(s, 'transcript', ''))[:90]!r}"
                        for s in segs
                    ),
                    **metrics,
                })

    return pd.DataFrame(rows)


EVAL_DETAIL_DF = build_video_eval_cache()
EVAL_SUMMARY_DF = summarize_video_eval(EVAL_DETAIL_DF)

display(EVAL_SUMMARY_DF)
display(EVAL_DETAIL_DF.head())


[InternVideo2 shared video-text space | Dense retrieval] Q1: Describe the overall architecture of the Transformer model
[InternVideo2 shared video-text space | Dense retrieval] Q2: Why did Transformers improve on earlier RNN and LSTM sequence models?
[InternVideo2 shared video-text space | Dense retrieval] Q3: What different roles do the attention layer and the MLP layer play in a Transformer block?
[InternVideo2 shared video-text space | Dense retrieval] Q4: Why is positional information added to Transformer input embeddings?
[InternVideo2 shared video-text space | Dense retrieval] Q5: How do queries, keys, and values determine which information a token receives in attention?
[InternVideo2 shared video-text space | Dense retrieval] Q6: How is attention computed efficiently for all tokens in parallel?
[InternVideo2 shared video-text space | Dense retrieval] Q7: Which attention variants does the video mention, and what broad purposes do they serve?
[InternVideo2 shared video-text space 

,approach,retrieval_view,bertscore_f1,precision,recall,context_recall,evidence_coverage,must_recall,should_recall,n_retrieved,retrieved_visuals
0,InternVideo2 shared video-text space,Dense + cross-encoder rerank,0.790,0.296,0.343,0.437,0.364,0.369,0.286,5.0,0.0
1,InternVideo2 shared video-text space,Dense retrieval,0.779,0.202,0.422,0.415,0.382,0.607,0.357,5.0,0.0
2,VLM frame summaries + Whisper + BGE,BM25 retrieval,0.834,0.312,0.651,0.905,0.881,0.905,0.714,5.0,5.0
3,VLM frame summaries + Whisper + BGE,Dense + BM25 + cross-encoder rerank,0.828,0.312,0.666,0.976,0.976,0.976,0.643,5.0,5.0
4,VLM frame summaries + Whisper + BGE,Dense + BM25 hybrid,0.822,0.334,0.678,0.976,0.952,1.000,0.714,5.0,5.0
5,VLM frame summaries + Whisper + BGE,Dense retrieval,0.823,0.337,0.684,0.976,0.952,1.000,0.714,5.0,5.0


,question_id,approach_key,approach,retrieval_view_key,retrieval_view,requires_video_evidence,question,expected_answer,answer,sources,...,context_recall,evidence_coverage,must_recall,should_recall,context_claims_covered,evidence_claims_covered,must_covered,should_covered,n_retrieved,retrieved_visuals
0,1,iv2,InternVideo2 shared video-text space,dense,Dense retrieval,True,Describe the overall architecture of the Trans...,The Transformer uses an encoder-decoder archit...,"Based on the provided transcript excerpts, her...",[04:00 – 04:08] | visual='' | speech=''\n[03:5...,...,0.333333,0.500000,1.0,0.5,2/6,3/6,4/4,1/2,5,0
1,2,iv2,InternVideo2 shared video-text space,dense,Dense retrieval,True,Why did Transformers improve on earlier RNN an...,Earlier RNNs and LSTMs process tokens one at a...,The provided transcript excerpts do not contai...,[05:30 – 05:38] | visual='' | speech='At each ...,...,0.400000,0.400000,0.0,0.0,2/5,2/5,0/4,0/1,5,0
2,3,iv2,InternVideo2 shared video-text space,dense,Dense retrieval,True,What different roles do the attention layer an...,The attention layer is the communication step:...,Based on the provided transcript excerpts:\n\n...,[03:36 – 03:44] | visual='' | speech=''\n[06:0...,...,1.000000,0.800000,1.0,0.5,5/5,4/5,3/3,1/2,5,0
3,4,iv2,InternVideo2 shared video-text space,dense,Dense retrieval,True,Why is positional information added to Transfo...,A Transformer has no sense of token order by d...,"[06:14 – 06:28]\n""Now reach context-aware repr...",[04:00 – 04:08] | visual='' | speech=''\n[03:5...,...,0.600000,0.400000,0.5,0.0,3/5,2/5,1/2,0/3,5,0
4,5,iv2,InternVideo2 shared video-text space,dense,Dense retrieval,True,"How do queries, keys, and values determine whi...","Each token produces a query, a key, and a valu...",Based on the provided transcript excerpt at [0...,[03:00 – 03:08] | visual='' | speech=''\n[02:5...,...,0.571429,0.571429,1.0,1.0,4/7,4/7,6/6,1/1,5,0


### 9.5 Interactive Check Table

This cell does not call retrieval or generation. It only filters `EVAL_DETAIL_DF`.


In [ ]:
import ipywidgets as widgets
from IPython.display import display


if "EVAL_DETAIL_DF" not in globals() or EVAL_DETAIL_DF is None or EVAL_DETAIL_DF.empty:
    raise RuntimeError("Run the previous 'Generate Evaluation Answers' cell before opening the table.")


# Detach callbacks and close widgets from a previous execution of this cell.
# This keeps one live dashboard even when the notebook cell is rerun.
_previous_video_dashboard = globals().get("_VIDEO_EVAL_DASHBOARD")
if _previous_video_dashboard:
    for widget in _previous_video_dashboard.get("observed_widgets", []):
        widget.unobserve(_previous_video_dashboard["callback"], names="value")
    _previous_video_dashboard["out"].clear_output(wait=False)
    _previous_video_dashboard["controls"].close()
    _previous_video_dashboard["out"].close()


approach_checks = {
    key: widgets.Checkbox(value=True, description=cfg["label"], indent=False)
    for key, cfg in VIDEO_EVAL_APPROACHES.items()
}

view_dropdowns = {
    key: widgets.Dropdown(
        options=[(label, view_key) for view_key, label in cfg["views"].items()],
        value=list(cfg["views"].keys())[-1],
        description="",
        layout=widgets.Layout(width="360px"),
    )
    for key, cfg in VIDEO_EVAL_APPROACHES.items()
}

question_slider = widgets.IntSlider(
    value=min(EVAL_QUESTION_LIMIT, int(EVAL_DETAIL_DF["question_id"].max())),
    min=1,
    max=int(EVAL_DETAIL_DF["question_id"].max()),
    step=1,
    description="Questions",
    continuous_update=False,
)
show_bertscore = widgets.Checkbox(value=True, description="BERTScore")
show_answers = widgets.Checkbox(value=False, description="Show answers")
out = widgets.Output()
_video_refresh_state = {"running": False}


def _filtered_video_eval() -> pd.DataFrame:
    frames = []
    max_q = question_slider.value
    for key, checkbox in approach_checks.items():
        if not checkbox.value:
            continue
        selected_view = view_dropdowns[key].value
        part = EVAL_DETAIL_DF[
            (EVAL_DETAIL_DF["approach_key"] == key)
            & (EVAL_DETAIL_DF["retrieval_view_key"] == selected_view)
            & (EVAL_DETAIL_DF["question_id"] <= max_q)
        ]
        frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def refresh_eval_table(*_):
    if _video_refresh_state["running"]:
        return
    _video_refresh_state["running"] = True
    try:
        # Use the widget-native method: IPython.clear_output can append output
        # in some Jupyter frontends when called from an observer callback.
        out.clear_output(wait=False)
        with out:
            df = _filtered_video_eval()
            if df.empty:
                print("No rows selected.")
                return

            summary = summarize_video_eval(df)
            detail_cols = [
                "question_id",
                "approach",
                "retrieval_view",
                "requires_video_evidence",
                "bertscore_f1",
                "precision",
                "recall",
                "context_recall",
                "evidence_coverage",
                "must_covered",
                "should_covered",
                "context_claims_covered",
                "evidence_claims_covered",
                "question",
            ]
            if not show_bertscore.value:
                summary = summary.drop(columns=["bertscore_f1"], errors="ignore")
                detail_cols.remove("bertscore_f1")
            if show_answers.value:
                detail_cols += ["answer", "expected_answer", "sources"]

            display(summary)
            display(df[detail_cols].round(3))
    finally:
        _video_refresh_state["running"] = False


observed_widgets = list(approach_checks.values()) + list(view_dropdowns.values()) + [
    question_slider,
    show_bertscore,
    show_answers,
]
for widget in observed_widgets:
    widget.observe(refresh_eval_table, names="value")

rows = []
for key, cfg in VIDEO_EVAL_APPROACHES.items():
    rows.append(widgets.HBox([
        approach_checks[key],
        widgets.Label("Retrieval view", layout=widgets.Layout(width="110px")),
        view_dropdowns[key],
    ]))

controls = widgets.VBox([
    widgets.HTML("<h3>Interactive Video RAG Evaluation Table</h3>"),
    *rows,
    widgets.HBox([question_slider, show_bertscore, show_answers]),
])

_VIDEO_EVAL_DASHBOARD = {
    "callback": refresh_eval_table,
    "observed_widgets": observed_widgets,
    "controls": controls,
    "out": out,
}

display(controls, out)
refresh_eval_table()


Output()

### 9.6 Final Mean Comparison Table

This widget summarizes `EVAL_DETAIL_DF` by averaging every metric over the selected evaluated questions. It compares every cached approach/retrieval-view combination directly and does not rerun retrieval, generation, or BERTScore.


In [ ]:
import ipywidgets as widgets
from IPython.display import display


VIDEO_SUMMARY_METRICS = [
    "bertscore_f1", "precision", "recall",
    "context_recall", "evidence_coverage", "must_recall", "should_recall",
    "n_retrieved", "retrieved_visuals",
]
VIDEO_SUMMARY_FORMAT = {
    "bertscore_f1": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "context_recall": "{:.3f}",
    "evidence_coverage": "{:.3f}",
    "must_recall": "{:.3f}",
    "should_recall": "{:.3f}",
    "n_retrieved": "{:.1f}",
    "retrieved_visuals": "{:.1f}",
}


def _video_mean_summary(detail_df, selected_keys, selected_views, question_limit, sort_metric, descending=True):
    if detail_df is None or detail_df.empty:
        return pd.DataFrame()
    df = detail_df.copy()
    if selected_keys:
        df = df[df["approach_key"].isin(selected_keys)]
    if selected_views:
        df = df[df["retrieval_view"].isin(selected_views)]
    if "question_id" in df.columns:
        df = df[df["question_id"] <= question_limit]
    if df.empty:
        return pd.DataFrame()

    metric_cols = [c for c in VIDEO_SUMMARY_METRICS if c in df.columns]
    grouped = df.groupby(["approach", "retrieval_view"], dropna=False)
    summary = grouped[metric_cols].mean(numeric_only=True).reset_index()
    counts = grouped["question_id"].nunique().reset_index(name="n_questions")
    summary = summary.merge(counts, on=["approach", "retrieval_view"], how="left")
    if sort_metric in summary.columns:
        summary = summary.sort_values(sort_metric, ascending=not descending, na_position="last")
    summary = summary.reset_index(drop=True)
    summary.insert(0, "rank", range(1, len(summary) + 1))
    ordered_cols = ["rank", "approach", "retrieval_view", "n_questions"] + metric_cols
    return summary[[c for c in ordered_cols if c in summary.columns]]


def _close_previous_video_summary_dashboard():
    previous = globals().get("_VIDEO_FINAL_SUMMARY_DASHBOARD")
    if not previous:
        return
    for widget in previous.get("observed_widgets", []):
        widget.unobserve(previous["callback"], names="value")
    previous["button"].on_click(previous["callback"], remove=True)
    previous["output"].clear_output(wait=False)
    previous["ui"].close()
    previous["output"].close()


def make_video_mean_summary_dashboard(detail_df=None):
    global _VIDEO_FINAL_SUMMARY_DASHBOARD

    if detail_df is None:
        detail_df = globals().get("EVAL_DETAIL_DF")
    if detail_df is None or detail_df.empty:
        print("Run the 'Generate Evaluation Answers' cell first to create EVAL_DETAIL_DF.")
        return None

    cached_keys = [key for key in VIDEO_EVAL_APPROACHES if key in set(detail_df["approach_key"])]
    approach_checks = {
        key: widgets.Checkbox(value=True, description=VIDEO_EVAL_APPROACHES[key]["label"], indent=False)
        for key in cached_keys
    }
    view_options = sorted(detail_df["retrieval_view"].dropna().unique().tolist())
    view_select = widgets.SelectMultiple(
        options=view_options,
        value=tuple(view_options),
        description="Views",
        rows=min(8, max(3, len(view_options))),
        layout=widgets.Layout(width="460px"),
    )
    max_questions = int(detail_df["question_id"].max())
    question_limit = widgets.IntSlider(
        value=max_questions,
        min=1,
        max=max_questions,
        step=1,
        description="Questions",
        continuous_update=False,
        layout=widgets.Layout(width="360px"),
    )
    metric_options = [(c.replace("_", " "), c) for c in VIDEO_SUMMARY_METRICS if c in detail_df.columns]
    default_metric = "bertscore_f1" if "bertscore_f1" in detail_df.columns else metric_options[0][1]
    sort_metric = widgets.Dropdown(options=metric_options, value=default_metric, description="Sort by")
    descending = widgets.Checkbox(value=True, description="Descending", indent=False)
    show_bert = widgets.Checkbox(value=True, description="Show BERTScore", indent=False)
    update_button = widgets.Button(description="Update summary", button_style="primary", icon="bar-chart")
    output = widgets.Output()
    render_state = {"running": False}

    def _render(_=None):
        if render_state["running"]:
            return
        render_state["running"] = True
        try:
            output.clear_output(wait=False)
            with output:
                selected_keys = [key for key, check in approach_checks.items() if check.value]
                if not selected_keys:
                    print("Select at least one approach.")
                    return
                selected_views = list(view_select.value)
                summary = _video_mean_summary(
                    detail_df,
                    selected_keys,
                    selected_views,
                    question_limit.value,
                    sort_metric.value,
                    descending.value,
                )
                if summary.empty:
                    print("No cached rows for this summary selection.")
                    return
                display_cols = list(summary.columns)
                if not show_bert.value:
                    display_cols = [c for c in display_cols if not c.startswith("bertscore")]
                display(summary[display_cols].style.format(VIDEO_SUMMARY_FORMAT, na_rep="n/a"))
        finally:
            render_state["running"] = False

    approach_box = widgets.VBox(list(approach_checks.values()))
    controls = widgets.HBox([
        approach_box,
        view_select,
        widgets.VBox([question_limit, sort_metric, widgets.HBox([descending, show_bert, update_button])]),
    ])
    observed_widgets = [
        *approach_checks.values(),
        view_select,
        question_limit,
        sort_metric,
        descending,
        show_bert,
    ]
    update_button.on_click(_render)
    for widget in observed_widgets:
        widget.observe(_render, names="value")

    ui = widgets.VBox([
        widgets.HTML("<h3>Final Mean Video RAG Comparison Table</h3>"),
        widgets.HTML("<i>Means are computed over the selected evaluated questions. No retrieval, generation, or BERTScore is rerun.</i>"),
        controls,
        output,
    ])
    _VIDEO_FINAL_SUMMARY_DASHBOARD = {
        "callback": _render,
        "observed_widgets": observed_widgets,
        "button": update_button,
        "output": output,
        "ui": ui,
    }
    display(ui)
    _render()
    return ui


_close_previous_video_summary_dashboard()
mean_summary_dashboard = make_video_mean_summary_dashboard()


### 9.7 Answer Generation Showcase


In [ ]:
SHOWCASE_QUESTIONS = [
    "Describe the overall architecture of the Transformer model",
    "Why is positional information added to Transformer input embeddings?",
    "How do queries, keys, and values determine which information a token receives in attention?",
]

def showcase_video_answer_generation(
    questions: Sequence[str] = SHOWCASE_QUESTIONS,
    approach_key: str = "unified",
    retrieval_view: str = "rerank",
) -> None:
    cfg = VIDEO_EVAL_APPROACHES[approach_key]
    print(f"Approach: {cfg['label']}")
    print(f"Retrieval: {cfg['views'][retrieval_view]}")
    print("=" * 90)

    for question in questions:
        segs = cfg["retrieve"](question, retrieval_view)
        answer = generate_answer_for_approach(approach_key, segs, question)
        print(f"Q: {question}\n")
        print(f"A: {answer}\n")
        print("Sources:")
        for seg in segs:
            print(f"  {getattr(seg, 'timestamp_label', '')}")
            visual = str(getattr(seg, "vision_summary", "")).strip()
            speech = str(getattr(seg, "transcript", "")).strip()
            if visual:
                print(f"    visual: {visual[:180]}")
            if speech:
                print(f"    speech: {speech[:180]}")
        print("-" * 90)


showcase_video_answer_generation()

Approach: VLM frame summaries + Whisper + BGE
Retrieval: Dense + BM25 + cross-encoder rerank
Q: Did you go to sleep

A: The provided context does not contain information about whether the speaker went to sleep or not.

Sources:
  [03:30 – 04:00]
    visual: The image shows a diagram from a presentation titled "Attention Is All You Need Explained." The diagram illustrates the architecture of a neural network model, specifically focusin
    speech: The transformer includes an encoder and a decoder. Both are made of stacked blocks. Each block has two key layers, an attention layer and a feedforward or MLP layer. The attention 
  [09:00 – 09:30]
    visual: The image shows a diagram explaining the concept of "Attention Is All You Need," which is a neural network architecture. The diagram includes various components such as Multi-Head 
    speech: the parameters that produce the queries, keys, and values are optimized. Over time, the attention layer learns meaningful patterns. For instance,